# RSNA Knee Abnormality Detection — Data Pipeline (v2)

Raw DICOM → **training-ready, memory-mapped tensors**, verified against the dataset audit. One code path serves the
offline cache, training and the inference notebook (train/inference parity by construction).

```
 raw DICOM ──► 1 INDEX ──► 2 SLOTS + LATERALITY ──► 3 LABELS + FOLDS ──► 4 CACHE ──► 5 QC ──► 6 LOADER ──► model
 (570 GB)      slice order   6 acquisition slots     gold + extra labels   uint8      gates     2.5D windows
               geometry      per study, side L/R     site-stratified       memmap     montage   + augmentation
```

| Stage | Entry point | Output | Cost (calibrated by cell 3) |
|---|---|---|---|
| 1-2 index | `runner.get_index` | `index.pkl`, `slots.pkl`, `sides.json`, `report.json` | one header pass over all files; checkpointed |
| 3 labels/folds | `labels.build_labels`, `splits.group_folds` | `labels.csv`, `folds_*.csv` | seconds |
| 4 cache | `runner.run_cache` | `<scratch>/knee_train.*` memmap | CPU/IO bound, **resumable**, workers write straight into the memmap |
| 5 QC | `runner.run_qc` | gates + `montage.png` | seconds |
| 6 loader | `RSNADataset` | `[S, W, 3, H, W]` uint8 per study | ~39 ms/sample/worker at 336 px (measured) |

## Why it is built this way (dataset fact → decision)

| Dataset fact (audit, all 24,371 series) | Decision in the pipeline |
|---|---|
| File names are random UIDs; `sorted(os.listdir)` is **not** anatomical order (P = 3.67% vs 3.51% expected for random) | slice order from `ImagePositionPatient` projected on the plane normal; sign-normalised so direction never depends on vendor IOP convention |
| One orientation code per plane in 100% of series (Sag `PI`, Cor `LI`, Ax `LP`) | assert + `reorient_stack` safety net (no-op on train) |
| Right and left knees both occur; no reliable tag | laterality = DICOM tag → patient-x of the image centre → unresolved; right knees mirrored to left (coronal/axial: columns; sagittal: slice order). **No flip augmentation** |
| 221 matrix sizes, FOV tight (median 160 mm), 7.3% non-square, 2:1 wide frames, FOV < crop exists | **physical** crop (130 mm) using each series' own row/column spacing; fg-centred for wide frames; border-median padding |
| p99 intensity spread 22×, heavy tails (8.2%), Canon negative pixels, Philips rescale up to 262 | stack-wide robust percentiles (P1–P99.5), negatives clipped, slope/intercept applied, never DICOM windows (vary per slice in 52%) |
| 837 3-D series (0.6–1 mm), 3.0% of public "most slices" picks are 3-D | z sampled on a **millimetre grid** (slab-mean for thin slices); 2-D preferred over 3-D |
| `Fluid_Sensitive` ≡ `Fat_Suppression` in train; descriptions 17.7% unusable; 5.4% of studies have stripped headers | public slot rule (header evidence) + CSV fallback only when the header carries no text |
| Hidden set differs from the placeholder (earlier public versions crashed on it) | nothing raises on data: every problem becomes a masked slot + a counter; empty studies get the median prediction, never a constant column |
| 98.7% of studies unlabeled; 58 gold | NaN → masked loss; optional extra (report-derived) labels with confidence weights; gold always wins |
| Group-by-scanner CV puts 205/238 stripped-header studies in one fold | **site-stratified** folds for model selection (every site in every fold) + group folds as a robustness check |

## Platform constraints this notebook adapts to (Kaggle docs / community reports)

* Sessions run up to **12 h** (CPU/GPU) → every long step is **checkpointed and resumable**.
* `/kaggle/working` is persisted output, capped at **~20 GB** (and a file-count limit) → only small artefacts go there
  (index, labels, QC); the big cache goes to **scratch** outside it (not saved between sessions; rebuilt in a new session).
* The scratch size is not documented → the notebook **measures free space** and picks the largest `(img_size, stack_depth)`
  that fits (`runner.fit_cache_cfg`). Full quality (336 px, 24 slices) needs ≈ 72 GB for the train set.
* 4 CPU cores / ~30 GB RAM → process pool pinned to cores, OpenCV threads off inside workers, cache accessed through
  memmap (no 20+ GB arrays in RAM).
* **Build the cache before touching the GPU** (forking after CUDA initialisation is unsafe) — the cells below are ordered that way.

## Package source
Each cell writes one module under `BASE` (default `/kaggle/working/knee`, or `./knee` elsewhere). Edit a cell and re-run it to change a module.

In [ ]:
import os, sys

BASE = os.environ.get('KNEE_BASE', '/kaggle/working/knee' if os.path.isdir('/kaggle/working') else os.path.abspath('knee'))
os.makedirs(BASE, exist_ok=True)

def write_module(rel, src):
    p = os.path.join(BASE, 'src', rel)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    with open(p, 'w') as fh:
        fh.write(src)
    print(f'wrote src/{rel}  ({len(src.splitlines())} lines)')

In [ ]:
# ---- src/__init__.py
write_module('__init__.py', r'''"""RSNA Knee Abnormality Detection source package (v2 preprocessing layer)."""
''')

In [ ]:
# ---- src/config.py
write_module('config.py', r'''"""Central configuration.

Legacy constants (SEED, TARGETS, SLOTS, ...) are unchanged so existing code keeps working.
All preprocessing behaviour lives in PreCfg; two presets exist:

  'public' : bit-compatible re-implementation of the public 0.943 reader (for weights trained with it
             and for A/B ablations)
  'v2'     : dataset-verified upgrade (physical-mm z sampling, position ordering, stack-wide robust
             normalisation, border-median padding, area resampling, reorientation, wide-frame centring)
"""
from dataclasses import dataclass, replace

# ------------------------------------------------------------------ legacy training constants
SEED = 2026
EPOCHS = 10
BATCH_SIZE = 8
LR_HEAD = 0.001
LR_BACKBONE = 8e-06
UNFREEZE_LAST = 6
WEIGHT_DECAY = 0.02
TIME_BUDGET_HOURS = 8.0          # was referenced by train.py but never defined

AUG_ROT_DEG = 8.0
AUG_SCALE = 0.08
AUG_SHIFT = 0.05
AUG_INTENSITY = 0.1

TARGETS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
    'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's",
    'Contusion', 'Fracture',
]

# name, plane, fluid-weighted (PD/T2), fat-suppressed
SLOTS = [
    ('SAG_FLUID_FS', 'Sagittal', True, True),
    ('COR_FLUID_FS', 'Coronal', True, True),
    ('AX_FLUID_FS', 'Axial', True, True),
    ('SAG_FLUID_NOFS', 'Sagittal', True, False),
    ('COR_T1', 'Coronal', False, False),
    ('SAG_T1', 'Sagittal', False, False),
]
N_SLOTS = len(SLOTS)

# ------------------------------------------------------------------ dataset facts (verified by the audit)
# One in-plane orientation code per plane in 100% of the 24,371 train series (and the test example).
# First letter = direction image COLUMNS increase toward, second = direction image ROWS increase toward (LPS).
CANON_ORIENT = {'Sagittal': 'PI', 'Coronal': 'LI', 'Axial': 'LP'}
LAT_MIN_OFFSET_MM = 20.0         # |patient-x of image centre| needed to call laterality from geometry
COMPETITION = 'rsna-knee-abnormality-detection'
ROOT_CANDIDATES = [
    f'/kaggle/input/competitions/{COMPETITION}',
    f'/kaggle/input/{COMPETITION}',
]


@dataclass(frozen=True)
class PreCfg:
    name: str = 'v2'
    # ---- in-plane
    crop_mm: float = 130.0
    img_size: int = 336
    pad_mode: str = 'border_median'      # 'border_median' | 'public' (no pad: skip crop if window > frame)
    resize_mode: str = 'area'            # 'area' (antialiased down / linear up) | 'bilinear' (public)
    wide_center: bool = True             # centre on foreground when frame aspect > wide_ratio
    wide_ratio: float = 1.3
    # ---- intensity
    norm_lo: float = 1.0
    norm_hi: float = 99.5
    clip_negative: bool = True
    # ---- through-plane
    order_mode: str = 'position'         # 'position' (sign-normalised) | 'normal' (public: dot(ipp, r x c))
    z_mode: str = 'mm'                   # 'mm' physical grid | 'index' (public linspace over band)
    stack_depth: int = 24                # slices stored per slot
    z_step_mm: float = 4.0               # spacing between stored slices (z_mode='mm')
    band: tuple = (0.2, 0.8)             # used when z_mode='index'
    # ---- windows (2.5D)
    group: int = 3                       # adjacent slices per window (channels)
    win_stride: int = 2
    # ---- canonicalisation
    reorient: bool = True                # map to CANON_ORIENT
    lat_canon: bool = True               # mirror right knees to left
    # ---- slot logic
    slot_prefer_2d: bool = True
    slot_fs_priority: str = 'hdr'        # 'hdr' (public rule, header evidence first) | 'csv'
    slot_csv_fallback: bool = True       # use CSV flags when the header carries no usable text

    @property
    def n_windows(self) -> int:
        return max(1, (self.stack_depth - self.group) // self.win_stride + 1)


PRESETS = {
    'public': PreCfg(name='public', pad_mode='public', resize_mode='bilinear', wide_center=False,
                     norm_lo=1.0, norm_hi=99.0, clip_negative=False, order_mode='normal', z_mode='index',
                     stack_depth=12, band=(0.2, 0.8), win_stride=1, reorient=False, slot_prefer_2d=False,
                     slot_csv_fallback=False),
    'v2': PreCfg(name='v2'),
}


def get_cfg(name: str = 'v2', **overrides) -> PreCfg:
    base = PRESETS[name]
    return replace(base, **overrides) if overrides else base


# legacy aliases (kept for old imports)
CROP_MM = PRESETS['v2'].crop_mm
IMG_SIZE = PRESETS['v2'].img_size
GROUP_SIZE = PRESETS['v2'].group


def cfg_from_dict(d):
    """Rebuild a PreCfg from a JSON-loaded dict (lists -> tuples)."""
    d = dict(d)
    if 'band' in d:
        d['band'] = tuple(d['band'])
    return PreCfg(**d)
''')

In [ ]:
# ---- src/preprocess/__init__.py
write_module('preprocess/__init__.py', r'''"""Dataset-verified preprocessing layer (numpy/OpenCV only; torch-free)."""
''')

In [ ]:
# ---- src/preprocess/dicomio.py
write_module('preprocess/dicomio.py', r'''"""Robust DICOM header / pixel access.

Dataset facts this module is built around (verified on all 24,371 train series):
  * every file is uncompressed Explicit VR LE, but the hidden test set is not guaranteed to be -> decode errors
    are caught per slice, never fatal
  * 26% of series are signed int16, Canon series contain negative pixels (down to -1871)
  * RescaleSlope/Intercept are constant inside a series (Philips slope reaches 262)
  * DICOM WindowCenter/Width vary per slice in 52% of series -> never used
"""
import os
import numpy as np
import pydicom

ORDER_TAGS = ['ImagePositionPatient', 'ImageOrientationPatient', 'InstanceNumber',
              'Rows', 'Columns', 'PixelSpacing']

REP_TAGS = ['SeriesDescription', 'SequenceName', 'ScanOptions', 'ScanningSequence', 'RepetitionTime', 'EchoTime',
            'InversionTime', 'MRAcquisitionType', 'Laterality', 'ImageLaterality', 'SliceThickness',
            'SpacingBetweenSlices', 'Manufacturer', 'ManufacturerModelName', 'MagneticFieldStrength',
            'BitsStored', 'PixelRepresentation', 'PhotometricInterpretation', 'BodyPartExamined']


def f1(v):
    """First float of a scalar / multi-value, else None."""
    try:
        return float(v)
    except Exception:
        try:
            return float(list(v)[0])
        except Exception:
            return None


def floats(v, n):
    try:
        a = [float(x) for x in v]
    except Exception:
        return None
    return a if len(a) == n and all(np.isfinite(a)) else None


def sval(v):
    """Header value -> plain str ('|' joined for multi-values), None if absent."""
    if v is None:
        return None
    if isinstance(v, (list, tuple)) or v.__class__.__name__ == 'MultiValue':
        return '|'.join(str(x) for x in v)
    return str(v)


def read_tags(path, tags):
    try:
        ds = pydicom.dcmread(path, stop_before_pixels=True, force=True, specific_tags=list(tags))
    except Exception:
        return None
    return ds


def representative_header(path):
    """Full (pixel-less) header of one file -> dict of REP_TAGS (strings) + numeric helpers."""
    try:
        ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)
    except Exception:
        return None
    out = {t: sval(getattr(ds, t, None)) for t in REP_TAGS}
    ps = floats(getattr(ds, 'PixelSpacing', None), 2)
    out['ps_row'], out['ps_col'] = (ps[0], ps[1]) if ps else (np.nan, np.nan)
    out['transfer_syntax'] = str(getattr(getattr(ds, 'file_meta', None), 'TransferSyntaxUID', ''))
    return out


def decode_slice(path):
    """Decode one slice -> float32 2-D array with RescaleSlope/Intercept applied.

    Raises on any failure (caller decides the fallback). MONOCHROME1 is inverted. Non-finite values -> 0.
    """
    ds = pydicom.dcmread(path, force=True)
    a = ds.pixel_array
    if a.ndim != 2:
        raise ValueError(f'expected a 2-D slice, got ndim={a.ndim}')
    a = a.astype(np.float32)
    slope = f1(getattr(ds, 'RescaleSlope', None))
    icpt = f1(getattr(ds, 'RescaleIntercept', None))
    a = a * (slope if slope else 1.0) + (icpt if icpt else 0.0)
    if str(getattr(ds, 'PhotometricInterpretation', '')).upper() == 'MONOCHROME1':
        a = a.max() - a
    if not np.isfinite(a).all():
        a = np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0)
    return a


def list_dicoms(series_dir):
    try:
        with os.scandir(series_dir) as it:
            return sorted(e.name for e in it if e.name.lower().endswith('.dcm'))
    except OSError:
        return []


def decode_raw(path):
    """Fast decode: RAW integer pixels (no float conversion) + rescale params.

    The float conversion is deferred until after the physical crop (crops cut ~40% of the pixels for the
    typical 130 mm window), which is where most of the per-slice CPU time goes. -> (raw 2-D, slope, intercept)
    MONOCHROME1 is inverted in the integer domain."""
    ds = pydicom.dcmread(path, force=True)
    a = ds.pixel_array
    if a.ndim != 2:
        raise ValueError(f'expected a 2-D slice, got ndim={a.ndim}')
    if str(getattr(ds, 'PhotometricInterpretation', '')).upper() == 'MONOCHROME1':
        a = a.max() - a
    slope = f1(getattr(ds, 'RescaleSlope', None)) or 1.0
    icpt = f1(getattr(ds, 'RescaleIntercept', None)) or 0.0
    return a, float(slope), float(icpt)
''')

In [ ]:
# ---- src/preprocess/geometry.py
write_module('preprocess/geometry.py', r'''"""Slice ordering, orientation, laterality.

Why this exists (verified on the dataset): file names are pydicom-generated random UIDs, so neither
`sorted(os.listdir())` nor any filename order carries anatomical position (observed P(first file == spatial
centre) = 3.67% vs 3.51% expected for random order, z = 1.36, on all 24,371 series). Slice order must come from
ImagePositionPatient projected on the slice normal.
"""
from collections import Counter
import numpy as np

from .. import config

_LET = {(0, 1): 'L', (0, -1): 'R', (1, 1): 'P', (1, -1): 'A', (2, 1): 'S', (2, -1): 'I'}
_AXIS = {'L': 0, 'R': 0, 'P': 1, 'A': 1, 'S': 2, 'I': 2}
PLANES = ('Sagittal', 'Coronal', 'Axial')


# --------------------------------------------------------------------------- orientation
def orient_info(iop):
    """-> (plane from slice normal, |normal| alignment with its axis, 2-letter LPS code).

    Code letter 1 = direction image COLUMNS increase toward; letter 2 = direction image ROWS increase toward.
    Dataset standard: Sagittal 'PI', Coronal 'LI', Axial 'LP'.
    """
    if iop is None or len(iop) != 6:
        return None, float('nan'), None
    r, c = np.asarray(iop[:3], float), np.asarray(iop[3:], float)
    n = np.cross(r, c)
    nn = np.linalg.norm(n)
    if not np.isfinite(nn) or nn < 1e-6:
        return None, float('nan'), None
    n = n / nn
    plane = PLANES[int(np.argmax(np.abs(n)))]
    code = ''
    for v in (r, c):
        ax = int(np.argmax(np.abs(v)))
        code += _LET[(ax, 1 if v[ax] >= 0 else -1)]
    return plane, float(np.abs(n).max()), code


def slice_normal(iop):
    r, c = np.asarray(iop[:3], float), np.asarray(iop[3:], float)
    n = np.cross(r, c)
    return n / (np.linalg.norm(n) + 1e-12)


def center_xyz(ipp, iop, ps, rows, cols):
    """Patient-space centre of a slice (mm)."""
    ipp, iop = np.asarray(ipp, float), np.asarray(iop, float)
    return ipp + iop[:3] * ps[1] * (cols - 1) / 2.0 + iop[3:] * ps[0] * (rows - 1) / 2.0


# --------------------------------------------------------------------------- ordering
def order_records(recs, order_mode='position'):
    """Order one series' slices by anatomical position.

    recs: list of dicts {name, ipp, iop, inst, rows, cols, ps}  (None where a tag is missing)
    order_mode:
      'position' : key = sign(n[axis]) * dot(ipp, n), ascending  -> increasing along +L / +P / +S
                   (direction is independent of the vendor's IOP sign convention)
      'normal'   : key = dot(ipp, r x c) ascending (public-notebook convention, no sign normalisation)
    Falls back to InstanceNumber, then natural name order, and reports which one was used.
    Files whose matrix differs from the majority shape are dropped (and counted).
    Returns dict(ordered, pos, method, n_dropped_shape, plane, code, align, spacing_med, z_extent, shape, ps,
                 center_x, rep_index) or None when there are no readable files.
    """
    recs = [r for r in recs if r is not None]
    if not recs:
        return None
    shapes = Counter((r['rows'], r['cols']) for r in recs)
    shape = shapes.most_common(1)[0][0]
    n_drop = sum(1 for r in recs if (r['rows'], r['cols']) != shape)
    recs = [r for r in recs if (r['rows'], r['cols']) == shape]
    iops = Counter(tuple(np.round(r['iop'], 3)) for r in recs if r['iop'] is not None)
    iop = np.array(iops.most_common(1)[0][0]) if iops else None
    plane, align, code = orient_info(iop) if iop is not None else (None, float('nan'), None)

    have_pos = [r for r in recs if r['ipp'] is not None]
    n = len(recs)
    nsign = 1.0
    # every branch builds tuples (key, tiebreak, name, rec); `pos` is in mm only for method == 'position'
    if iop is not None and len(have_pos) >= max(2, int(0.8 * n)):
        nrm = slice_normal(iop)
        ax = int(np.argmax(np.abs(nrm)))
        nsign = 1.0 if nrm[ax] >= 0 else -1.0
        sgn = nsign if order_mode == 'position' else 1.0
        spare = float(np.median([sgn * float(np.dot(r['ipp'], nrm)) for r in have_pos]))
        keyed = [(sgn * float(np.dot(r['ipp'], nrm)) if r['ipp'] is not None else spare,
                  r['inst'] if r['inst'] is not None else float('inf'), r['name'], r) for r in recs]
        method = 'position'
    elif sum(r['inst'] is not None for r in recs) >= max(2, int(0.8 * n)):
        keyed = [(r['inst'] if r['inst'] is not None else float('inf'), 0.0, r['name'], r) for r in recs]
        method = 'instance'
    else:
        keyed = [(0.0, 0.0, r['name'], r) for r in recs]
        method = 'name'
    keyed.sort(key=lambda t: (t[0], t[1], t[2]))
    ordered = [k[2] for k in keyed]
    pos = np.array([k[0] for k in keyed], dtype=np.float64) if method != 'name' else np.arange(n, dtype=np.float64)
    dp = np.diff(pos) if len(pos) > 1 else np.array([])
    spacing = float(np.median(np.abs(dp))) if len(dp) else float('nan')
    rep = len(ordered) // 2
    rep_rec = keyed[rep][3]
    ps = rep_rec['ps'] if rep_rec['ps'] else (float('nan'), float('nan'))
    cx = float('nan')
    if rep_rec['ipp'] is not None and iop is not None and rep_rec['ps']:
        try:
            cx = float(center_xyz(rep_rec['ipp'], iop, rep_rec['ps'], shape[0], shape[1])[0])
        except Exception:
            cx = float('nan')
    return dict(ordered=ordered, pos=pos, method=method, n_dropped_shape=n_drop, plane=plane, code=code,
                align=align, spacing_med=spacing, z_extent=float(pos[-1] - pos[0]) if len(pos) > 1 else 0.0,
                shape=shape, ps=ps, center_x=cx, rep_index=rep, nsign=nsign)


# --------------------------------------------------------------------------- laterality
def _side_from_tag(v):
    if v is None:
        return None
    s = str(v).strip().upper()
    return s[0] if s and s[0] in ('L', 'R') else None


def resolve_laterality(rows, min_offset=None):
    """Study-level side of the knee.

    rows: iterable of dicts with optional 'Laterality', 'ImageLaterality' (strings) and 'center_x' (mm).
    Priority: DICOM tag (majority vote) -> patient-x of the image centre (median over series, |x| >= min_offset:
    x<0 -> Right, else Left) -> unresolved ('U').
    Returns dict(side in {'L','R','U'}, source, x_med, disagree).
    """
    min_offset = config.LAT_MIN_OFFSET_MM if min_offset is None else min_offset
    tags = []
    xs = []
    for r in rows:
        for k in ('Laterality', 'ImageLaterality'):
            s = _side_from_tag(r.get(k))
            if s:
                tags.append(s)
        x = r.get('center_x')
        if x is not None and np.isfinite(x):
            xs.append(float(x))
    xm = float(np.median(xs)) if xs else float('nan')
    geo = None
    if np.isfinite(xm) and abs(xm) >= min_offset:
        geo = 'R' if xm < 0 else 'L'
    if tags:
        side = Counter(tags).most_common(1)[0][0]
        return dict(side=side, source='tag', x_med=xm, disagree=bool(geo and geo != side))
    if geo:
        return dict(side=geo, source='geometry', x_med=xm, disagree=False)
    return dict(side='U', source='none', x_med=xm, disagree=False)


# --------------------------------------------------------------------------- canonicalisation
def reorient_stack(stack, code, plane):
    """Map a [D,H,W] stack to the dataset-standard orientation code of `plane` (CANON_ORIENT).

    Handles axis swaps (transpose) and sign flips. A no-op for all 24,371 train series (single code per plane)."""
    target = config.CANON_ORIENT.get(plane)
    if not code or len(code) != 2 or not target or code == target:
        return stack, False
    r, c = code
    tr, tc = target
    if _AXIS[r] != _AXIS[tr]:
        stack = stack.transpose(0, 2, 1)
        r, c = c, r
    if r != tr:
        stack = stack[:, :, ::-1]
    if c != tc:
        stack = stack[:, ::-1, :]
    return np.ascontiguousarray(stack), True


def canonical_side(stack, plane, side):
    """Mirror right knees so they look like left knees.

    Coronal/Axial: flip image columns. Sagittal: reverse slice order (slices run medial<->lateral)."""
    if side != 'R':
        return stack
    if plane in ('Coronal', 'Axial'):
        return np.ascontiguousarray(stack[:, :, ::-1])
    return np.ascontiguousarray(stack[::-1])
''')

In [ ]:
# ---- src/preprocess/index.py
write_module('preprocess/index.py', r'''"""Series index: one row per series on disk, with anatomical slice order, geometry and a representative header.

Design rules (from the dataset audit + the hidden-rerun failures documented in the public notebooks):
  * the DIRECTORY TREE is the truth for which series/files exist; the CSVs only contribute flags
    (test_series.csv in the hidden set may differ from the placeholder -> never assume it is complete)
  * a failure inside one series never aborts the run: the row is kept with `err` set and an empty file list
  * the build is chunked + checkpointed, so a 40-minute scan survives a session restart
"""
import os
import time
import pickle
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

import numpy as np
import pandas as pd

from .. import config
from . import dicomio as dio
from . import geometry as geo

SPLIT_DIRS = {'train': 'train_series', 'test': 'test_series'}


def discover_root(explicit=None):
    """Find the competition folder (handles /kaggle/input/competitions/<slug> and /kaggle/input/<slug>)."""
    cands = [explicit, os.environ.get('KNEE_DATA')] + list(config.ROOT_CANDIDATES)
    for c in cands:
        if c and os.path.isdir(c) and (os.path.isdir(os.path.join(c, 'train_series')) or
                                       os.path.isdir(os.path.join(c, 'test_series'))):
            return c
    base = '/kaggle/input'
    if os.path.isdir(base):
        for p in sorted(os.listdir(base)):
            for q in (os.path.join(base, p), *[os.path.join(base, p, x) for x in sorted(os.listdir(os.path.join(base, p)))[:20]
                                                 if os.path.isdir(os.path.join(base, p, x))]):
                if os.path.isdir(os.path.join(q, 'test_series')) or os.path.isdir(os.path.join(q, 'train_series')):
                    return q
    raise FileNotFoundError('competition root not found; pass it explicitly or set KNEE_DATA')


def list_series_dirs(root, split):
    base = os.path.join(root, SPLIT_DIRS[split])
    items = []
    if not os.path.isdir(base):
        return items
    with os.scandir(base) as it:
        studies = sorted(e.name for e in it if e.is_dir())
    for st in studies:
        sd = os.path.join(base, st)
        with os.scandir(sd) as it2:
            for e in sorted(it2, key=lambda x: x.name):
                if e.is_dir():
                    items.append((split, st, e.name, e.path))
    return items


def scan_series(item):
    """Full slice ordering + representative header for one series directory."""
    split, study, series, d = item
    row = dict(split=split, StudyInstanceUID=study, SeriesInstanceUID=series, dir=d, n_files=0,
               ordered_files=[], positions=np.zeros(0), err=None)
    try:
        names = dio.list_dicoms(d)
        row['n_files'] = len(names)
        recs, bad = [], 0
        for nm in names:
            ds = dio.read_tags(os.path.join(d, nm), dio.ORDER_TAGS)
            if ds is None or 'Rows' not in ds or 'Columns' not in ds:
                bad += 1
                continue
            ps = dio.floats(getattr(ds, 'PixelSpacing', None), 2)
            recs.append(dict(name=nm, ipp=dio.floats(getattr(ds, 'ImagePositionPatient', None), 3),
                             iop=dio.floats(getattr(ds, 'ImageOrientationPatient', None), 6),
                             inst=dio.f1(getattr(ds, 'InstanceNumber', None)),
                             rows=int(ds.Rows), cols=int(ds.Columns), ps=tuple(ps) if ps else None))
        row['n_bad'] = bad
        o = geo.order_records(recs, 'position')
        if o is None:
            row['err'] = 'no_readable_slices'
            return row
        row.update(ordered_files=o['ordered'], positions=o['pos'].astype(np.float32), order_method=o['method'],
                   n_dropped_shape=o['n_dropped_shape'], plane_geo=o['plane'], orient_code=o['code'],
                   plane_align=o['align'], spacing_med=o['spacing_med'], z_extent=o['z_extent'],
                   rows=o['shape'][0], cols=o['shape'][1], ps_row=o['ps'][0], ps_col=o['ps'][1],
                   center_x=o['center_x'], nsign=o['nsign'], n_slices=len(o['ordered']))
        rep = dio.representative_header(os.path.join(d, o['ordered'][o['rep_index']]))
        if rep:
            row.update({k: v for k, v in rep.items() if k not in ('ps_row', 'ps_col')})
            if not np.isfinite(row['ps_row']) and np.isfinite(rep['ps_row']):
                row['ps_row'], row['ps_col'] = rep['ps_row'], rep['ps_col']
    except Exception as e:                                     # never fatal
        row['err'] = f'{type(e).__name__}: {str(e)[:80]}'
    return row


def build_index(root, splits=('train', 'test'), workers=None, chunk=1000, cache_dir=None, limit=0, force=False,
                progress=True):
    """Scan every series on disk. Returns a DataFrame (object columns: ordered_files, positions)."""
    workers = workers or max(2, (os.cpu_count() or 4) * 2)
    items = []
    for sp in splits:
        items += list_series_dirs(root, sp)
    if limit:
        items = items[:limit]
    chunks = [items[i:i + chunk] for i in range(0, len(items), chunk)]
    if cache_dir:                                    # stale-checkpoint guard: chunks are only valid for THIS item list
        import glob
        import hashlib
        sig = hashlib.md5(('|'.join(i[3] for i in items) + f'#{chunk}').encode()).hexdigest()
        sp = os.path.join(cache_dir, 'signature.txt')
        os.makedirs(cache_dir, exist_ok=True)
        if not (os.path.exists(sp) and open(sp).read() == sig):
            for f in glob.glob(os.path.join(cache_dir, 'index_*.pkl')):
                os.remove(f)
            open(sp, 'w').write(sig)
    parts, t0 = [], time.time()
    ex = None
    try:
        ex = ProcessPoolExecutor(max_workers=workers, mp_context=mp.get_context('fork'))
    except Exception:
        ex = ThreadPoolExecutor(max_workers=workers)
    for k, ch in enumerate(chunks):
        p = os.path.join(cache_dir, f'index_{k:04d}.pkl') if cache_dir else None
        if p and os.path.exists(p) and not force:
            parts.append(pd.read_pickle(p))
            continue
        df = pd.DataFrame(list(ex.map(scan_series, ch, chunksize=8)))
        if p:
            os.makedirs(cache_dir, exist_ok=True)
            df.to_pickle(p)
        parts.append(df)
        if progress:
            print(f'  index chunk {k + 1}/{len(chunks)}  {(k + 1) * chunk / max(time.time() - t0, 1e-9):.1f} series/s', flush=True)
    ex.shutdown()
    out = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    return out


def attach_csv_flags(index_df, root):
    """Left-join the competition CSV flags (plane / fluid / fat-sat) onto the on-disk index."""
    frames = []
    for name in ('train_series.csv', 'test_series.csv'):
        p = os.path.join(root, name)
        if os.path.exists(p):
            frames.append(pd.read_csv(p))
    if not frames:
        for c in ('Anatomical_Plane', 'Fluid_Sensitive', 'Fat_Suppression'):
            index_df[c] = np.nan if c != 'Anatomical_Plane' else None
        return index_df
    csv = pd.concat(frames, ignore_index=True).drop_duplicates('SeriesInstanceUID')
    keep = [c for c in ('SeriesInstanceUID', 'Anatomical_Plane', 'Fluid_Sensitive', 'Fat_Suppression') if c in csv.columns]
    out = index_df.merge(csv[keep], on='SeriesInstanceUID', how='left')
    for c in ('Anatomical_Plane', 'Fluid_Sensitive', 'Fat_Suppression'):
        if c not in out.columns:
            out[c] = np.nan if c != 'Anatomical_Plane' else None
    return out


def save_index(df, path):
    with open(path, 'wb') as fh:
        pickle.dump(df, fh, protocol=4)


def load_index(path):
    with open(path, 'rb') as fh:
        return pickle.load(fh)


def index_report(df):
    """Human-readable data-contract statistics (printed by tools/build_index.py)."""
    rep = {}
    ok = df[df.err.isna()] if 'err' in df else df
    rep['series'] = len(df)
    rep['series_with_error'] = int(df.err.notna().sum()) if 'err' in df else 0
    rep['empty_series'] = int((df.n_files == 0).sum())
    if 'order_method' in ok:
        rep['order_method'] = ok.order_method.value_counts().to_dict()
    if 'orient_code' in ok and 'plane_geo' in ok:
        rep['orient_codes'] = {f'{p}:{c}': int(n) for (p, c), n in ok.groupby('plane_geo').orient_code.value_counts().items()}
        bad = 0
        for pl, code in config.CANON_ORIENT.items():
            sub = ok[ok.plane_geo == pl]
            bad += int((sub.orient_code != code).sum())
        rep['non_canonical_orientation'] = bad
    if 'Anatomical_Plane' in ok:
        m = ok[ok.Anatomical_Plane.notna() & ok.plane_geo.notna()]
        rep['csv_plane_vs_geometry_mismatch'] = int((m.Anatomical_Plane != m.plane_geo).sum())
    if 'n_dropped_shape' in ok:
        rep['series_with_mixed_shapes'] = int((ok.n_dropped_shape > 0).sum())
    if 'n_bad' in ok:
        rep['series_with_unreadable_slices'] = int((ok.n_bad > 0).sum())
    return rep
''')

In [ ]:
# ---- src/preprocess/slots.py
write_module('preprocess/slots.py', r'''"""Series -> acquisition-slot assignment.

Annotation constants are the public 0.943 notebook's (regression-tested against it on all 4,407 train studies:
SAG/COR/AX fluid-FS slots filled for 94.3 / 95.5 / 99.4 % of studies).

What the audit added:
  * the CSV `Fat_Suppression` flag agrees with the header-derived fat-sat evidence for 99.14% of series, and
    `Fluid_Sensitive` is IDENTICAL to it in train (it tracks fat-suppression, not PD/T2 contrast), so the CSV fluid
    flag is only used as a last-resort proxy;
  * the metadata-poor cohort (238 studies, no TR/TE/ScanOptions) still has descriptions, so header rules work;
  * hidden-set robustness: every missing input degrades to a weaker signal instead of an exception;
  * 3.0% of 'most slices' picks are 3-D volumes -> `prefer_2d` picks a 2-D series whenever one exists.
"""
import re
import numpy as np
import pandas as pd

from .. import config

FATSAT_OPTS = {'FS', 'FATSAT', 'FAT_SAT', 'FSAT'}
_FATSAT_RX = re.compile('\\bfs\\b|fatsat|fat sat|\\bstir\\b|\\bspair\\b|\\bspir\\b|\\bwe\\b|water excit|\\btirm\\b|'
                        '\\bsting\\b|\\bfatsup\\b|smart fat|\\bwater\\b')
_T1_RX = re.compile('\\bt1\\b|\\bt1w\\b')
_T2_RX = re.compile('\\bt2\\b|\\bt2w\\b')
_PD_RX = re.compile('\\bpd\\b|\\bpdw\\b|proton|\\bdp\\b|dens')
_SEP = re.compile('[_\\-.]')

NEEDED = ['SeriesDescription', 'SequenceName', 'ScanOptions', 'ScanningSequence', 'RepetitionTime', 'EchoTime']


def to_flag(x):
    """Robust 0/1/NaN from CSV values (hidden-set CSV formatting is not guaranteed)."""
    if x is None:
        return np.nan
    if isinstance(x, (bool, np.bool_)):
        return float(bool(x))
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x) if x in (0, 1) else np.nan
    s = str(x).strip().lower()
    if s in ('1', 'true', 'yes', 'y', 't', '1.0'):
        return 1.0
    if s in ('0', 'false', 'no', 'n', 'f', '0.0'):
        return 0.0
    return np.nan


def annotate(df, fs_priority='hdr', csv_fallback=True):
    """Add fatsat / weight / fluid / informative / is3d columns. Returns a copy.

    fatsat: header evidence (description regex or ScanOptions token) as in the public pipeline. When the header
    has no usable text at all (dummy/missing description AND no ScanOptions) the CSV flag is used instead; with
    fs_priority='csv' the CSV flag wins whenever present.
    """
    df = df.copy()
    for c in NEEDED:
        if c not in df.columns:
            df[c] = None
    raw = df['SeriesDescription'].fillna('').astype(str)
    dummy = raw.str.contains('dummyseriesdesc', case=False)
    desc = (raw.where(~dummy, '') + ' ' + df['SequenceName'].fillna('').astype(str)).str.lower()
    desc = desc.str.replace(_SEP, ' ', regex=True)
    opts = df['ScanOptions'].fillna('').astype(str).str.upper().str.split('|')
    opts_fs = opts.apply(lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
    fs_hdr = desc.str.contains(_FATSAT_RX) | opts_fs
    informative = (desc.str.strip() != '') | (df['ScanOptions'].fillna('').astype(str).str.strip() != '')

    csv_fs = df['Fat_Suppression'].map(to_flag) if 'Fat_Suppression' in df.columns else pd.Series(np.nan, index=df.index)
    if fs_priority == 'csv':
        fs = np.where(csv_fs.notna(), csv_fs == 1.0, fs_hdr)
        src = np.where(csv_fs.notna(), 'csv', 'hdr')
    else:
        use_csv = ((~informative) & csv_fs.notna()) if csv_fallback else pd.Series(False, index=df.index)
        fs = np.where(use_csv, csv_fs == 1.0, fs_hdr)
        src = np.where(use_csv, 'csv', 'hdr')
    df['fatsat'] = fs.astype(bool)
    df['fs_source'] = src
    df['fs_conflict'] = csv_fs.notna() & ((csv_fs.fillna(0).astype(float) == 1.0) != df['fatsat'])

    tr = pd.to_numeric(df['RepetitionTime'], errors='coerce')
    te = pd.to_numeric(df['EchoTime'], errors='coerce')
    gre = df['ScanningSequence'].fillna('').astype(str).str.upper().str.contains('GR')
    t1, t2, pdw = desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX)
    w = np.where(t1 & ~t2 & ~pdw, 'T1', np.where(t2 & ~pdw, 'T2', np.where(pdw, 'PD', np.where(
        gre, 'GRE', np.where(tr < 800, 'T1', np.where(te > 60, 'T2', np.where(tr >= 800, 'PD', 'UNK')))))))
    df['weight'] = w
    fluid = np.isin(w, ['PD', 'T2'])
    # last resort: no header contrast at all -> the CSV fluid flag (== fat-sat flag in train) as a proxy
    if csv_fallback and 'Fluid_Sensitive' in df.columns:
        cf = df['Fluid_Sensitive'].map(to_flag)
        unk = (w == 'UNK') & cf.notna()
        fluid = np.where(unk, cf == 1.0, fluid)
    df['fluid'] = fluid.astype(bool)

    acq = df['MRAcquisitionType'].fillna('').astype(str).str.upper() if 'MRAcquisitionType' in df.columns else ''
    ns = pd.to_numeric(df['n_slices'] if 'n_slices' in df.columns else df['n_files'], errors='coerce').fillna(0)
    th = pd.to_numeric(df.get('SliceThickness', np.nan), errors='coerce')
    df['is3d'] = (acq == '3D') | ((ns > 120) & (th < 1.5))
    return df


def plane_of(df):
    """CSV plane (verified 100% = DICOM geometry) with geometry as the fallback."""
    p = df['Anatomical_Plane'] if 'Anatomical_Plane' in df.columns else pd.Series(None, index=df.index)
    g = df['plane_geo'] if 'plane_geo' in df.columns else pd.Series(None, index=df.index)
    ok = p.isin(['Sagittal', 'Coronal', 'Axial'])
    return p.where(ok, g)


def assign_slots(g, prefer_2d=True, slots=None):
    """g: annotated rows of ONE study (needs plane, fatsat, fluid, is3d, n_slices, SeriesInstanceUID).

    Returns (chosen, alts, extras):
      chosen: {slot_name: SeriesInstanceUID}
      alts:   {slot_name: [other candidates of the same slot type, best first]}   (view-swap augmentation pool)
      extras: series not used by any slot
    Rule (public): most slices wins. With prefer_2d a 2-D series always beats a 3-D volume.
    """
    slots = slots or config.SLOTS
    chosen, alts, used = {}, {}, set()
    gg = g.assign(_n=pd.to_numeric(g['n_slices'], errors='coerce').fillna(0))
    for name, plane, fluid, fs in slots:
        sel = (gg['plane'] == plane) & (gg['fatsat'] == fs)
        if fluid is not None:
            sel &= (gg['fluid'] == fluid)
        c = gg[sel]
        if len(c) == 0:
            continue
        if prefer_2d:
            c = c.assign(_k=c['is3d'].astype(int))
            c = c.sort_values(['_k', '_n', 'SeriesInstanceUID'], ascending=[True, False, True])
        else:
            c = c.sort_values(['_n', 'SeriesInstanceUID'], ascending=[False, True])
        chosen[name] = c.iloc[0]['SeriesInstanceUID']
        alts[name] = [s for s in c['SeriesInstanceUID'].tolist()[1:]]
        used.add(chosen[name])
    extras = [s for s in gg['SeriesInstanceUID'] if s not in used]
    return chosen, alts, extras


def assign_all(index_df, prefer_2d=True, fs_priority='hdr', csv_fallback=True):
    """Annotate the whole index and assign slots for every study (fully vectorised: ~O(n log n), no per-study loop).

    Returns (annotated_df, slot_table):
      slot_table: DataFrame indexed by StudyInstanceUID, one column per slot holding a SeriesInstanceUID or None;
      slot_table.attrs['alts']: {slot: {study: [alternate SeriesInstanceUIDs]}} for view-swap augmentation.
    Tie-break is deterministic (series UID), unlike a stable sort over directory order."""
    df = index_df.copy()
    df['plane'] = plane_of(df)
    df = annotate(df, fs_priority, csv_fallback)
    ns = pd.to_numeric(df['n_slices'] if 'n_slices' in df.columns else df['n_files'], errors='coerce').fillna(0)
    usable = df['plane'].isin(['Sagittal', 'Coronal', 'Axial']) & (ns > 0)
    work = df.assign(_n=ns, _k=(df['is3d'].astype(int) if prefer_2d else 0))[usable]
    work = work.sort_values(['StudyInstanceUID', '_k', '_n', 'SeriesInstanceUID'], ascending=[True, True, False, True])
    studies = pd.Index(df['StudyInstanceUID'].unique(), name='StudyInstanceUID')
    tab = pd.DataFrame(index=studies, columns=[n for n, *_ in config.SLOTS], dtype=object)
    alts = {}
    for name, plane, fluid, fs in config.SLOTS:
        m = (work['plane'] == plane) & (work['fatsat'] == fs)
        if fluid is not None:
            m &= (work['fluid'] == fluid)
        c = work[m]
        first = ~c.duplicated('StudyInstanceUID')
        tab.loc[c.loc[first, 'StudyInstanceUID'].values, name] = c.loc[first, 'SeriesInstanceUID'].values
        rest = c[~first]
        alts[name] = rest.groupby('StudyInstanceUID')['SeriesInstanceUID'].apply(list).to_dict() if len(rest) else {}
    tab = tab.where(tab.notna(), None)
    tab.attrs['alts'] = alts
    return df, tab
''')

In [ ]:
# ---- src/preprocess/pixels.py
write_module('preprocess/pixels.py', r'''"""Pixel pipeline: decode -> physical crop -> robust normalise -> resample -> uint8 stack.

Throughput design (each choice measured/justified in tools/bench.py):
  * decode RAW integers, crop FIRST, convert only the cropped window to float32
  * slab averaging of thin 3-D slices happens at crop resolution, once per target slice
  * percentiles on a 2x2-subsampled stack (exact percentiles for the 'public' preset)
  * cv2 resize (SIMD) instead of per-slice torch calls; cv2/BLAS threads pinned to 1 inside workers
Dataset-driven rules (audit): physical mm crop with each series' own row/column spacing (7.3% of matrices are
non-square, 0.39% have non-square pixels, 640x1280 wide frames exist), border-median padding (72% of series have a
non-zero noise floor), negatives clipped (241 Canon series), per-series robust scale (22x spread), heavy tails
(8.2% of series have p99.9 > 2x p99).
"""
import os
import numpy as np
import cv2

from . import dicomio as dio

cv2.setNumThreads(0)                    # parallelism comes from worker processes; avoid oversubscription
PS_FALLBACK = 0.3125                    # dataset median in-plane spacing (mm), only if the header has none


# --------------------------------------------------------------------------- z sampling
def z_groups(pos, cfg, spacing, method='position'):
    """Which native slices feed each of the cfg.stack_depth stored slices.

    Returns (groups: list[list[int]], valid: bool[D]).
      'index' (public): linspace over cfg.band of the ordered slice indices, duplicates padded
      'mm': physical grid cfg.z_step_mm centred on the series; nearest native slice, or the mean of all native
            slices inside +-step/2 when the native spacing is finer than 0.75*step (3-D / thin-slice series)
    """
    n = len(pos)
    D = cfg.stack_depth
    if cfg.z_mode == 'index' or method != 'position':
        # public linspace over the band; also the safe fallback when slice order came from InstanceNumber/name
        # (positions are then index units, not mm, so a millimetre grid would be meaningless)
        band = cfg.band if cfg.z_mode == 'index' else (0.0, 1.0)
        lo, hi = int(band[0] * (n - 1)), int(band[1] * (n - 1))
        idx = np.unique(np.linspace(lo, hi, D).astype(int)) if hi > lo else np.array([n // 2])
        while len(idx) < D:
            idx = np.append(idx, idx[-1])
        return [[int(i)] for i in idx[:D]], np.ones(D, bool)
    pos = np.asarray(pos, np.float64)
    step = cfg.z_step_mm
    mid = 0.5 * (pos[0] + pos[-1])
    targets = mid + (np.arange(D) - (D - 1) / 2.0) * step
    sp = spacing if np.isfinite(spacing) and spacing > 0 else step
    tol = 0.75 * max(sp, step)
    near = np.clip(np.searchsorted(pos, targets), 1, max(n - 1, 1))
    if n == 1:
        near = np.zeros(D, int)
    else:
        left, right = pos[near - 1], pos[near]
        near = np.where(np.abs(targets - left) <= np.abs(right - targets), near - 1, near)
    dist = np.abs(pos[near] - targets)
    valid = dist <= tol
    groups = []
    slab = sp < 0.75 * step
    for t, k in zip(targets, near):
        if slab:
            lo_i = int(np.searchsorted(pos, t - step / 2.0, 'left'))
            hi_i = int(np.searchsorted(pos, t + step / 2.0, 'right'))
            groups.append(list(range(lo_i, hi_i)) if hi_i > lo_i else [int(k)])
        else:
            groups.append([int(k)])
    return groups, valid


# --------------------------------------------------------------------------- crop geometry
def _border_median(a):
    h, w = a.shape
    b = max(2, int(0.02 * min(h, w)))
    ring = np.concatenate([a[:b].ravel(), a[-b:].ravel(), a[b:-b, :b].ravel(), a[b:-b, -b:].ravel()])
    return float(np.median(ring))


def _fg_center(raws):
    """Foreground bbox centre (row, col) in full-frame pixels, from the mean of the decoded slices."""
    ref = np.mean([np.maximum(a[::4, ::4].astype(np.float32), 0) for a in raws], axis=0)
    p99 = np.percentile(ref, 99)
    if p99 <= 0:
        return None
    fg = ref > 0.1 * p99
    if not fg.any():
        return None
    rr, cc = np.where(fg.any(1))[0], np.where(fg.any(0))[0]
    return 4.0 * (rr[0] + rr[-1]) / 2.0, 4.0 * (cc[0] + cc[-1]) / 2.0


def crop_plan(shape, ps_row, ps_col, cfg, raws=None):
    """-> (y0, x0, Hc, Wc) of the physical crop window in full-frame pixels (may exceed the frame -> padded),
    or None when the public rule skips the crop."""
    H, W = shape
    ok = np.isfinite(ps_row) and np.isfinite(ps_col) and ps_row > 0 and ps_col > 0
    if cfg.pad_mode == 'public':                                   # public rule, verbatim
        if not (ok and ps_row > 0):
            return None
        want = int(round(cfg.crop_mm / ps_row))
        if 16 < want < min(H, W):
            cy, cx = H // 2, W // 2
            half = want // 2
            return max(0, cy - half), max(0, cx - half), min(H, cy + half) - max(0, cy - half), \
                min(W, cx + half) - max(0, cx - half)
        return None
    if not ok:
        ps_row = ps_col = PS_FALLBACK
    Hc = max(16, int(round(cfg.crop_mm / ps_row)))
    Wc = max(16, int(round(cfg.crop_mm / ps_col)))
    cy, cx = (H - 1) / 2.0, (W - 1) / 2.0
    aspect = (W * ps_col) / (H * ps_row)
    if cfg.wide_center and raws and (aspect > cfg.wide_ratio or aspect < 1.0 / cfg.wide_ratio):
        c = _fg_center(raws)
        if c is not None:
            if aspect > 1:
                cx = c[1]
            else:
                cy = c[0]
    y0 = int(round(cy - (Hc - 1) / 2.0))
    x0 = int(round(cx - (Wc - 1) / 2.0))
    return y0, x0, Hc, Wc


def _crop_float(raw, slope, icpt, plan, bg_fn):
    """Crop (with border-median padding) and convert ONLY the window to float32."""
    H, W = raw.shape
    if plan is None:
        return raw.astype(np.float32) * slope + icpt
    y0, x0, Hc, Wc = plan
    ys0, xs0, ys1, xs1 = max(y0, 0), max(x0, 0), min(y0 + Hc, H), min(x0 + Wc, W)
    win = raw[ys0:ys1, xs0:xs1].astype(np.float32) * slope + icpt
    if (ys1 - ys0, xs1 - xs0) == (Hc, Wc):
        return win
    out = np.full((Hc, Wc), bg_fn(raw, slope, icpt), np.float32)
    out[ys0 - y0:ys1 - y0, xs0 - x0:xs1 - x0] = win
    return out


# --------------------------------------------------------------------------- normalise / resize
def normalise(vol, valid, cfg):
    """Stack-wide robust scaling to [0,1] (all slices share one window; invalid slices untouched)."""
    if cfg.clip_negative:
        np.maximum(vol, 0, out=vol)
    v = vol[valid]
    sub = v if cfg.name == 'public' else v[:, ::2, ::2]
    lo, hi = np.percentile(sub, [cfg.norm_lo, cfg.norm_hi])
    out = np.clip((vol - lo) / max(float(hi - lo), 1e-6), 0.0, 1.0)
    return out


def resize_stack(vol, size, mode):
    """[D,h,w] float -> [D,size,size] float. 'area' = antialiased when shrinking, bilinear when enlarging."""
    D, h, w = vol.shape
    if mode == 'area' and (h > size or w > size):
        interp = cv2.INTER_AREA
    else:
        interp = cv2.INTER_LINEAR
    out = np.empty((D, size, size), np.float32)
    for i in range(D):
        out[i] = cv2.resize(vol[i], (size, size), interpolation=interp)
    return out


def to_uint8(x):
    return np.clip(np.rint(x * 255.0), 0, 255).astype(np.uint8)


# --------------------------------------------------------------------------- one series -> stack
def build_stack(rec, cfg):
    """rec: index row (dict-like) with dir, ordered_files, positions, nsign, ps_row, ps_col, spacing_med.

    -> (uint8 [D,S,S], valid bool [D], info) or (None, None, info) when nothing could be decoded.
    Never raises: every slice failure is recorded in info['errors']."""
    info = dict(errors=[], padded=False, ps_missing=False, n_decoded=0)
    files = list(rec['ordered_files'])
    pos = np.asarray(rec['positions'], np.float64)
    if not files:
        info['errors'].append('no_files')
        return None, None, info
    if cfg.order_mode == 'normal' and float(rec.get('nsign', 1.0)) < 0:     # public sign convention
        files, pos = files[::-1], -pos[::-1]
    method = str(rec.get('order_method', 'position'))
    groups, valid = z_groups(pos, cfg, float(rec.get('spacing_med', np.nan)), method)
    info['z_fallback'] = cfg.z_mode == 'mm' and method != 'position'
    need = sorted({i for g in groups for i in g})
    raws = {}
    for i in need:
        try:
            raws[i] = dio.decode_raw(os.path.join(rec['dir'], files[i]))
        except Exception as e:
            info['errors'].append(f'{files[i]}: {type(e).__name__}')
    info['n_decoded'] = len(raws)
    if not raws:
        return None, None, info
    shape = max({r[0].shape for r in raws.values()}, key=lambda s: sum(r[0].shape == s for r in raws.values()))
    raws = {i: r for i, r in raws.items() if r[0].shape == shape}
    ps_row, ps_col = float(rec.get('ps_row', np.nan)), float(rec.get('ps_col', np.nan))
    info['ps_missing'] = not (np.isfinite(ps_row) and np.isfinite(ps_col) and ps_row > 0 and ps_col > 0)
    plan = crop_plan(shape, ps_row, ps_col, cfg, [r[0] for r in raws.values()] if cfg.wide_center else None)
    if plan is not None:
        info['padded'] = plan[0] < 0 or plan[1] < 0 or plan[0] + plan[2] > shape[0] or plan[1] + plan[3] > shape[1]
    def _bg(raw, slope, icpt):
        return _border_median(raw.astype(np.float32) * slope + icpt)

    D = len(groups)
    crops = {}
    for i, (raw, slope, icpt) in raws.items():
        crops[i] = _crop_float(raw, slope, icpt, plan, _bg)
    ref_i = next(iter(crops))
    vol = np.zeros((D,) + crops[ref_i].shape, np.float32)
    ok = np.zeros(D, bool)
    for d, g in enumerate(groups):
        got = [crops[i] for i in g if i in crops]
        if not got:
            continue
        vol[d] = got[0] if len(got) == 1 else np.mean(got, axis=0, dtype=np.float32)
        ok[d] = True
    valid = valid & ok
    if cfg.z_mode == 'index' or method != 'position':   # failed slices borrow the nearest decoded neighbour
        for d in np.where(~ok)[0]:
            j = int(np.argmin(np.where(ok, np.abs(np.arange(D) - d), 10 ** 6)))
            vol[d] = vol[j]
        valid = np.ones(D, bool)
    if not valid.any():
        return None, None, info
    vol = normalise(vol, valid, cfg)
    vol = resize_stack(vol, cfg.img_size, cfg.resize_mode)
    out = to_uint8(vol)
    out[~valid] = 0
    return out, valid, info
''')

In [ ]:
# ---- src/preprocess/sampling.py
write_module('preprocess/sampling.py', r'''"""Window selection + augmentation (the training hot loop; numpy/OpenCV only).

Throughput rules:
  * choose the windows FIRST, then augment only the slices those windows use (training uses ~4 of 11 windows,
    i.e. ~12 of the 24 stored slices -> ~2x less warping)
  * one affine matrix and one 256-entry LUT per slot, applied with cv2 (SIMD) to all of its slices
  * no flips: knees are canonicalised to 'left' at preprocessing time, so a horizontal flip would destroy the
    medial/lateral image-side consistency that the slot head relies on
"""
import numpy as np
import cv2

cv2.setNumThreads(0)


def n_windows(depth, group, stride):
    return max(1, (depth - group) // stride + 1)


def window_starts(depth, group, stride, n_use=None, train=False, rng=None):
    """Start indices (into the stored stack) of the windows to use.

    eval : evenly spaced subset (or all)           train: one random window per equal-width bin (stratified, so
    the whole depth is covered every step) with a random phase."""
    W = n_windows(depth, group, stride)
    starts = np.arange(W) * stride
    if n_use is None or n_use >= W:
        return starts
    if train:
        rng = rng or np.random.default_rng()
        edges = np.linspace(0, W, n_use + 1)
        pick = [int(rng.integers(int(np.floor(edges[i])), max(int(np.floor(edges[i])) + 1, int(np.ceil(edges[i + 1]))))) for i in range(n_use)]
        return starts[np.clip(pick, 0, W - 1)]
    return starts[np.unique(np.rint(np.linspace(0, W - 1, n_use)).astype(int))]


def gather_windows(stack, valid, starts, group):
    """stack [D,H,W] (may be a memmap view) -> windows [n,group,H,W] uint8 and window validity [n].

    A window is valid only if all of its channels are valid slices."""
    idx = np.asarray(starts)[:, None] + np.arange(group)[None, :]
    win = np.ascontiguousarray(stack[idx])
    wv = np.asarray(valid)[idx].all(axis=1)
    return win, wv


def _lut(rng, a):
    c = 1.0 + rng.uniform(-a, a)
    b = rng.uniform(-a / 2.0, a / 2.0)
    g = 1.0 + rng.uniform(-a, a)
    x = np.arange(256, dtype=np.float32) / 255.0
    y = np.clip((np.power(x, g) - 0.5) * c + 0.5 + b, 0.0, 1.0)
    return np.rint(y * 255.0).astype(np.uint8)


def augment_slot(win, rng, rot_deg=8.0, scale=0.08, shift=0.05, intensity=0.1):
    """win [n,g,H,W] uint8 -> same shape. One random rotation/scale/shift + one intensity LUT for the whole slot."""
    n, g, H, W = win.shape
    ang = rng.uniform(-rot_deg, rot_deg)
    sc = 1.0 + rng.uniform(-scale, scale)
    M = cv2.getRotationMatrix2D(((W - 1) / 2.0, (H - 1) / 2.0), ang, sc)
    M[0, 2] += rng.uniform(-shift, shift) * W
    M[1, 2] += rng.uniform(-shift, shift) * H
    lut = _lut(rng, intensity) if intensity > 0 else None
    out = np.empty_like(win)
    for i in range(n):
        for j in range(g):
            x = cv2.warpAffine(win[i, j], M, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
            out[i, j] = cv2.LUT(x, lut) if lut is not None else x
    return out
''')

In [ ]:
# ---- src/preprocess/pipeline.py
write_module('preprocess/pipeline.py', r'''"""One code path for everything: offline cache build, training fallback, and the inference notebook.

prepare_series / prepare_study NEVER raise (hidden-rerun lesson: data-dependent aborts are what broke earlier
public versions). Every problem becomes an entry in `info` and a masked slot, never a crash or a NaN.
"""
import numpy as np

from .. import config
from . import geometry as geo
from . import pixels


def index_to_records(df):
    """DataFrame -> {SeriesInstanceUID: dict}. Done once; avoids pandas in the hot path."""
    return {r['SeriesInstanceUID']: r for r in df.to_dict('records')} if len(df) else {}


def study_sides(index_df):
    """{study: laterality dict} using the DICOM tag first, patient-x geometry second."""
    out = {}
    cols = [c for c in ('StudyInstanceUID', 'Laterality', 'ImageLaterality', 'center_x') if c in index_df.columns]
    for st, g in index_df[cols].groupby('StudyInstanceUID', sort=False):
        out[st] = geo.resolve_laterality(g.to_dict('records'))
    return out


def prepare_series(rec, plane, side, cfg):
    """-> (uint8 [D,S,S], valid [D], info) or (None, None, info)."""
    stack, valid, info = pixels.build_stack(rec, cfg)
    if stack is None:
        return None, None, info
    info['reoriented'] = False
    if cfg.reorient:
        stack, changed = geo.reorient_stack(stack, rec.get('orient_code'), plane)
        info['reoriented'] = bool(changed)          # (a transposition leaves the depth axis untouched)
    if cfg.lat_canon and side == 'R':
        stack = geo.canonical_side(stack, plane, 'R')
        if plane == 'Sagittal':
            valid = valid[::-1].copy()
        info['mirrored'] = True
    return stack, valid, info


def prepare_study(study, slot_row, records, side, cfg):
    """slot_row: {slot_name: SeriesInstanceUID or None}.

    -> dict(stack uint8 [S,D,H,W], valid bool [S,D], slot_mask uint8 [S], info {slot: info}, side)"""
    S, D, H = config.N_SLOTS, cfg.stack_depth, cfg.img_size
    stack = np.zeros((S, D, H, H), np.uint8)
    valid = np.zeros((S, D), bool)
    mask = np.zeros(S, np.uint8)
    infos = {}
    for k, (name, plane, _, _) in enumerate(config.SLOTS):
        uid = slot_row.get(name)
        if not uid or uid not in records:
            continue
        try:
            st, v, info = prepare_series(records[uid], plane, side, cfg)
        except Exception as e:                                   # belt and braces
            st, v, info = None, None, dict(errors=[f'{type(e).__name__}: {str(e)[:60]}'])
        infos[name] = info
        if st is None:
            continue
        stack[k], valid[k], mask[k] = st, v, 1
    return dict(stack=stack, valid=valid, slot_mask=mask, info=infos, side=side)
''')

In [ ]:
# ---- src/preprocess/cache.py
write_module('preprocess/cache.py', r'''"""Memory-mapped uint8 study cache + a parallel builder with no inter-process array traffic.

Layout (one contiguous 16 MB block per study at D=24, S=6, 336^2 -> one sequential read per sample):
    <prefix>.images.u8   [N, S, D, H, W] uint8        <prefix>.valid.u1  [N, S, D]
    <prefix>.slot.u1     [N, S]                       <prefix>.done.u1   [N]
    <prefix>.meta.json   study ids, cfg, shapes       <prefix>.meta.csv  per-study side/source/qc

Throughput design:
  * workers write straight into the shared memmap (MAP_SHARED) -> nothing but a tiny dict is pickled back
  * longest-processing-time-first scheduling (3-D series are ~5x a 2-D one) removes the straggler tail
  * resumable: `done` flags live in a memmap, so a killed run restarts where it stopped
  * optional streaming hook (`on_ready`) lets inference run the GPU on chunk k while CPUs build chunk k+1
"""
import os
import json
import time
import shutil
import multiprocessing as mp
from dataclasses import asdict
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd

from .. import config
from . import pipeline

_G = {}


def _paths(prefix):
    return {k: f'{prefix}.{v}' for k, v in dict(images='images.u8', valid='valid.u1', slot='slot.u1',
                                                 done='done.u1', meta='meta.json', csv='meta.csv').items()}


class StudyCache:
    def __init__(self, prefix, mode='r'):
        self.prefix, self.mode = prefix, mode
        p = _paths(prefix)
        with open(p['meta']) as fh:
            self.meta = json.load(fh)
        self.studies = self.meta['studies']
        N, S, D, H = self.meta['N'], self.meta['S'], self.meta['D'], self.meta['H']
        m = 'r+' if mode == 'r+' else 'r'
        self.images = np.memmap(p['images'], np.uint8, m, shape=(N, S, D, H, H))
        self.valid = np.memmap(p['valid'], np.uint8, m, shape=(N, S, D))
        self.slot = np.memmap(p['slot'], np.uint8, m, shape=(N, S))
        self.done = np.memmap(p['done'], np.uint8, m, shape=(N,))
        self.index = {s: i for i, s in enumerate(self.studies)}

    def __len__(self):
        return len(self.studies)

    @staticmethod
    def create(prefix, studies, cfg):
        N, S, D, H = len(studies), config.N_SLOTS, cfg.stack_depth, cfg.img_size
        need = N * S * D * H * H
        d = os.path.dirname(os.path.abspath(prefix))
        os.makedirs(d, exist_ok=True)
        free = shutil.disk_usage(d).free
        if need * 1.02 > free:
            raise OSError(f'cache needs {need / 1e9:.1f} GB but only {free / 1e9:.1f} GB free in {d}')
        p = _paths(prefix)
        np.memmap(p['images'], np.uint8, 'w+', shape=(N, S, D, H, H)).flush()
        np.memmap(p['valid'], np.uint8, 'w+', shape=(N, S, D)).flush()
        np.memmap(p['slot'], np.uint8, 'w+', shape=(N, S)).flush()
        np.memmap(p['done'], np.uint8, 'w+', shape=(N,)).flush()
        with open(p['meta'], 'w') as fh:
            json.dump(dict(studies=list(studies), N=N, S=S, D=D, H=H, cfg=asdict(cfg)), fh)
        return StudyCache(prefix, 'r+')

    def slot_stack(self, i, s):
        return self.images[i, s]

    def flush(self):
        for a in (self.images, self.valid, self.slot, self.done):
            a.flush()


def cfg_of(prefix):
    """The PreCfg a cache was built with (single source of truth for training/inference on that cache)."""
    with open(_paths(prefix)['meta']) as fh:
        return config.cfg_from_dict(json.load(fh)['cfg'])


def _work(i):
    g = _G
    st = g['studies'][i]
    res = pipeline.prepare_study(st, g['slot_rows'].get(st, {}), g['records'], g['sides'].get(st, {}).get('side', 'U'), g['cfg'])
    c = g['cache']
    c.images[i] = res['stack']
    c.valid[i] = res['valid']
    c.slot[i] = res['slot_mask']
    c.done[i] = 1
    errs = sum(len(v.get('errors', [])) for v in res['info'].values())
    pad = sum(bool(v.get('padded')) for v in res['info'].values())
    miss = sum(bool(v.get('ps_missing')) for v in res['info'].values())
    return i, int(res['slot_mask'].sum()), errs, pad, miss


def _cost(study, slot_rows, records):
    c = 0
    for uid in slot_rows.get(study, {}).values():
        r = records.get(uid) if uid else None
        if r:
            c += (r.get('n_slices') or 0) * (5 if str(r.get('MRAcquisitionType')) == '3D' else 1)
    return c


def build_cache(prefix, studies, slot_rows, records, sides, cfg, workers=None, resume=True, order='lpt',
                chunk=64, on_ready=None, progress=True):
    """Build (or resume) the cache. Returns the open StudyCache (mode r+) and a stats dict.

    studies: ordered list of StudyInstanceUID. slot_rows: {study: {slot: series uid|None}}.
    on_ready(cache, a, b): called in the main thread whenever studies [a,b) are all done (use order='seq')."""
    workers = workers or max(2, os.cpu_count() or 2)
    p = _paths(prefix)
    if resume and os.path.exists(p['meta']):
        cache = StudyCache(prefix, 'r+')
        if cache.studies != list(studies) or cache.meta['cfg'] != json.loads(json.dumps(asdict(cfg))):
            raise ValueError('existing cache was built for different studies/cfg; use resume=False or another prefix')
    else:
        cache = StudyCache.create(prefix, studies, cfg)
    todo = [i for i in range(len(studies)) if not cache.done[i]]
    if order == 'lpt':
        todo.sort(key=lambda i: -_cost(studies[i], slot_rows, records))
    _G.update(studies=list(studies), slot_rows=slot_rows, records=records, sides=sides, cfg=cfg, cache=cache)
    try:
        ex = ProcessPoolExecutor(max_workers=workers, mp_context=mp.get_context('fork'))
    except Exception:
        ex = ThreadPoolExecutor(max_workers=workers)
    t0 = time.time()
    stats = dict(studies=len(studies), built=0, resumed=len(studies) - len(todo), decode_errors=0, padded=0, ps_missing=0,
                 empty_studies=0)
    futs = {ex.submit(_work, i): i for i in todo}
    pending = {}
    for a in range(0, len(studies), chunk):
        pending[a] = sum(1 for i in range(a, min(a + chunk, len(studies))) if not cache.done[i])
    for a, left in list(pending.items()):
        if left == 0 and on_ready:
            on_ready(cache, a, min(a + chunk, len(studies)))
        if left == 0:
            pending.pop(a)
    for k, f in enumerate(as_completed(futs)):
        i, nslot, errs, pad, miss = f.result()
        stats['built'] += 1
        stats['decode_errors'] += errs
        stats['padded'] += pad
        stats['ps_missing'] += miss
        stats['empty_studies'] += int(nslot == 0)
        a = (i // chunk) * chunk
        if a in pending:
            pending[a] -= 1
            if pending[a] == 0:
                if on_ready:
                    on_ready(cache, a, min(a + chunk, len(studies)))
                pending.pop(a)
        if progress and (k + 1) % 200 == 0:
            print(f'  cache {k + 1}/{len(todo)}  {(k + 1) / (time.time() - t0):.1f} studies/s', flush=True)
    ex.shutdown()
    cache.flush()
    stats['seconds'] = time.time() - t0
    pd.DataFrame(dict(StudyInstanceUID=studies, side=[sides.get(s, {}).get('side', 'U') for s in studies],
                      side_source=[sides.get(s, {}).get('source', 'none') for s in studies],
                      n_slots=np.asarray(cache.slot).sum(1))).to_csv(p['csv'], index=False)
    return cache, stats
''')

In [ ]:
# ---- src/preprocess/qc.py
write_module('preprocess/qc.py', r'''"""Data-contract checks. Cheap enough to run on every build; their numbers are the acceptance gates."""
import numpy as np
import pandas as pd

from .. import config


def cache_sanity(cache, n_sample=200, seed=0):
    """Sample built studies and verify the cache is not silently broken (constant images, empty-but-flagged slots,
    all-invalid depth, wrong dtype)."""
    rng = np.random.default_rng(seed)
    done = np.where(np.asarray(cache.done) == 1)[0]
    if len(done) == 0:
        return dict(studies_checked=0)
    pick = rng.choice(done, size=min(n_sample, len(done)), replace=False)
    bad = dict(constant_slot=[], flagged_but_blank=[], low_valid_depth=[], unflagged_but_filled=[])
    n_slots = 0
    stds, means, vfrac = [], [], []
    for i in pick:
        for s in range(config.N_SLOTS):
            filled = bool(cache.slot[i, s])
            st = np.asarray(cache.images[i, s, ::2, ::4, ::4])          # cheap subsample
            if filled:
                n_slots += 1
                v = float(np.mean(cache.valid[i, s]))
                vfrac.append(v)
                if st.max() == 0:
                    bad['flagged_but_blank'].append((cache.studies[i], s))
                if st.std() < 1.0:
                    bad['constant_slot'].append((cache.studies[i], s))
                if v < 0.5:
                    bad['low_valid_depth'].append((cache.studies[i], s))
                stds.append(float(st.std()))
                means.append(float(st.mean()))
            elif st.max() > 0:
                bad['unflagged_but_filled'].append((cache.studies[i], s))
    out = dict(studies_checked=len(pick), slots_checked=n_slots,
               mean_valid_depth=float(np.mean(vfrac)) if vfrac else float('nan'),
               slot_mean_p5_p95=[float(np.percentile(means, 5)), float(np.percentile(means, 95))] if means else None,
               slot_std_p5_p95=[float(np.percentile(stds, 5)), float(np.percentile(stds, 95))] if stds else None)
    out.update({k: len(v) for k, v in bad.items()})
    out['failures'] = {k: v[:5] for k, v in bad.items() if v}
    return out


def laterality_report(sides):
    df = pd.DataFrame(sides).T
    rep = dict(studies=len(df), by_side=df['side'].value_counts().to_dict(), by_source=df['source'].value_counts().to_dict(),
               tag_geometry_disagree=int(df['disagree'].sum()))
    rep['unresolved_frac'] = float((df['side'] == 'U').mean())
    return rep
''')

In [ ]:
# ---- src/preprocess/text.py
write_module('preprocess/text.py', r'''"""Tiny text/vendor helpers used to build leakage-safe folds (no external NLP dependency)."""
import re

_STOP = {
    'en': 'the and of with is are there no without in to an at from which seen noted',
    'es': 'el la los las de del con sin se en y una un que es hay no observa',
    'pt': 'o os as de da do com sem em e uma um que não nao há ha',
    'fr': 'le la les des du de et avec sans un une est pas il y a en dans',
    'it': 'il lo gli le di del della con senza non è una un e nel nella si',
    'de': 'der die das und mit ohne kein keine ist nicht eine im den des zeigt',
    'tr': 've ile bir olarak yok izlendi görülmektedir normal de da bulunmaktadır',
    'hr/bs/sr': 'i u je na se bez s nema nije vidljiv vidljiva uredan uredna',
    'nl': 'de het een en met zonder geen is niet van in op te',
}
_STOP = {k: set(v.split()) for k, v in _STOP.items()}


def guess_lang(text):
    """Coarse report-language proxy (script ranges + stop-word scores). Only used as a SITE proxy for fold grouping,
    never as a feature. Known limitation: templated English reports can be tagged 'tr'; paired with scanner model
    in the group key, this does not matter."""
    if not isinstance(text, str) or len(text.strip()) < 5:
        return 'empty'
    t = text.lower()
    letters = [ch for ch in t if ch.isalpha()]
    if not letters:
        return 'empty'

    def frac(a, b):
        return sum(a <= ch <= b for ch in letters) / len(letters)
    if frac('\u0400', '\u04ff') > .3:
        return 'cyrillic'
    if frac('\u0370', '\u03ff') > .3:
        return 'greek'
    if frac('\u0e00', '\u0e7f') > .3:
        return 'thai'
    if frac('\u4e00', '\u9fff') > .2:
        return 'cjk'
    if frac('\u0600', '\u06ff') > .3:
        return 'arabic'
    toks = re.findall(r'[^\W\d_]+', t)
    sc = {k: sum(w in v for w in toks) for k, v in _STOP.items()}
    sc['tr'] += 2 * sum(ch in 'ğışİ' for ch in text) / max(len(toks), 1) * 10
    best = max(sc, key=sc.get)
    return best if sc[best] >= 3 else 'unknown'


def vendor_of(manufacturer):
    """Normalise the many spellings seen in the data (Siemens/SIEMENS/Siemens Healthineers, GE/GEHC, ...)."""
    m = str(manufacturer).lower()
    for k, v in (('siemens', 'SIEMENS'), ('ge medical', 'GE'), ('gehc', 'GE'), ('philips', 'PHILIPS'),
                 ('toshiba', 'CANON/TOSHIBA'), ('canon', 'CANON/TOSHIBA'), ('fuji', 'FUJI/HITACHI'),
                 ('hitachi', 'FUJI/HITACHI')):
        if k in m:
            return v
    return 'OTHER'
''')

In [ ]:
# ---- src/preprocess/splits.py
write_module('preprocess/splits.py', r'''"""Leakage-safe group folds.

Groups = (report language | scanner model): language is a strong site proxy in this dataset (e.g. every Cyrillic
report is a Philips Ingenia, every German/French/Greek/Dutch report is Siemens) and scanner model separates the
many sites hiding inside the 1,717 English reports. Audit numbers: 72 groups, largest 5.9% of studies, 24 groups
with <10 studies. Rare groups are merged into 'rare|<vendor>'. Known duplicate exams share a group.
Identical report TEXTS are deliberately NOT unioned into groups: templated English reports recur across sites and
would chain unrelated sites into one giant component.
"""
import numpy as np
import pandas as pd

# the only true duplicate exams found by the audit (5 identical series each, Siemens Aera, non-gold)
KNOWN_DUP_PAIRS = [(
    '1.2.826.0.1.3680043.8.498.99725523560254732094432244556623376711',
    '1.2.826.0.1.3680043.8.498.99926624968240681772735102303531613209',
)]


def make_groups(study_meta, min_group=10, dup_pairs=None, n_splits=5):
    sm = study_meta.copy()
    g = sm['lang'].fillna('?').astype(str) + '|' + sm['model'].fillna('?').astype(str)
    cnt = g.map(g.value_counts())
    g = g.where(cnt >= min_group, 'rare|' + sm['vendor'].fillna('?').astype(str))
    cnt = g.map(g.value_counts())                       # a 'rare|<vendor>' bucket can itself be too small to stratify
    g = g.where(cnt >= n_splits, 'rare|ALL')
    sm['group'] = g
    for a, b in (dup_pairs if dup_pairs is not None else KNOWN_DUP_PAIRS):
        if a in set(sm['StudyInstanceUID']) and b in set(sm['StudyInstanceUID']):
            ga = sm.loc[sm['StudyInstanceUID'] == a, 'group'].iloc[0]
            sm.loc[sm['StudyInstanceUID'] == b, 'group'] = ga
    return sm


def group_folds(study_meta, n_splits=5, seed=2026, min_group=10, dup_pairs=None, scheme='site'):
    """-> DataFrame[StudyInstanceUID, group, fold, ...].

    scheme='site'  (default, for model selection): every site/scanner group is spread evenly over all folds
                   (stratified by group) - mirrors the hidden test, which is drawn from the same sites as train.
                   Duplicate exams are kept together.
    scheme='group' (robustness check): whole groups are held out together = unseen-scanner generalisation.
                   Pessimistic by construction: audit shows 205/238 metadata-poor studies land in one fold and
                   most languages are absent from some training folds."""
    sm = make_groups(study_meta, min_group, dup_pairs, n_splits)
    fold = np.full(len(sm), -1)
    if len(sm) < 2 * n_splits:                           # tiny datasets (smoke tests): round-robin, still dup-safe
        order = np.random.default_rng(seed).permutation(len(sm))
        fold[order] = np.arange(len(sm)) % n_splits
        sm['fold'] = fold
        return sm[['StudyInstanceUID', 'group', 'fold'] + [c for c in ('lang', 'model', 'vendor', 'gold') if c in sm.columns]]
    if scheme == 'group':
        from sklearn.model_selection import StratifiedGroupKFold
        y = sm['vendor'].fillna('?').astype('category').cat.codes.values
        for k, (_, va) in enumerate(StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed).split(sm, y, sm['group'])):
            fold[va] = k
    elif scheme == 'site':
        from sklearn.model_selection import StratifiedKFold
        uid = sm['StudyInstanceUID'].values
        partner = {}
        for a, b in (dup_pairs if dup_pairs is not None else KNOWN_DUP_PAIRS):
            partner[b] = a                                           # b follows a
        unit = np.array([partner.get(u, u) for u in uid])
        first = ~pd.Series(unit).duplicated().values                 # one representative row per unit
        rep = sm[first].reset_index(drop=True)
        y = rep['group'].astype('category').cat.codes.values
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        ufold = {}
        for k, (_, va) in enumerate(skf.split(rep, y)):
            for u in rep.loc[va, 'StudyInstanceUID']:
                ufold[u] = k
        fold = np.array([ufold[u] for u in unit])
    else:
        raise ValueError(scheme)
    sm['fold'] = fold
    return sm[['StudyInstanceUID', 'group', 'fold'] + [c for c in ('lang', 'model', 'vendor', 'gold') if c in sm.columns]]


def make_study_meta(index_df, train_csv=None, labels_df=None):
    """Study-level table (vendor, scanner model, report language, gold flag, metadata-poor flag) straight from the
    index + train.csv, so the notebook never needs the audit's study_meta.csv."""
    from .text import guess_lang, vendor_of
    df = index_df[index_df['split'] == 'train'] if 'split' in index_df.columns else index_df
    mode = lambda s: s.dropna().mode().iloc[0] if s.notna().any() else np.nan
    g = df.groupby('StudyInstanceUID')
    sm = pd.DataFrame({
        'vendor': g['Manufacturer'].agg(lambda s: mode(s.map(vendor_of))) if 'Manufacturer' in df else '?',
        'model': g['ManufacturerModelName'].agg(mode) if 'ManufacturerModelName' in df else '?',
        'poor': g['RepetitionTime'].agg(lambda s: s.isna().all()) if 'RepetitionTime' in df else False,
    }).reset_index()
    sm['lang'] = '?'
    if train_csv is not None and 'Report' in train_csv.columns:
        lang = train_csv.set_index('StudyInstanceUID')['Report'].map(guess_lang)
        sm['lang'] = sm['StudyInstanceUID'].map(lang).fillna('?')
    if labels_df is not None:
        from .. import config
        if 'source' in labels_df.columns:                       # build_labels() marks gold / extra / none
            has = labels_df['source'] == 'gold'
        else:
            has = labels_df[[t for t in config.TARGETS if t in labels_df.columns]].notna().any(axis=1)
        gold = labels_df.assign(_g=has).set_index('StudyInstanceUID')['_g']
        sm['gold'] = sm['StudyInstanceUID'].map(gold).fillna(False)
    else:
        sm['gold'] = False
    return sm
''')

In [ ]:
# ---- src/preprocess/loader.py
write_module('preprocess/loader.py', r'''"""Cache -> training/eval sample (numpy only, so it is testable without torch; dataset.py just wraps it)."""
import numpy as np

from .. import config
from . import sampling


def make_sample(cache, i, cfg, train=False, n_use=None, rng=None, aug=True):
    """One study from the memmap cache.

    Returns imgs uint8 [S, W, G, H, W], slot_mask uint8 [S], win_mask bool [S, W].
    train: independent stratified-random windows per slot + one geometric/intensity augmentation per slot.
    eval : the same evenly spaced windows for every slot (n_use=None -> all windows)."""
    S, G, H = config.N_SLOTS, cfg.group, cfg.img_size
    D, stride = cfg.stack_depth, cfg.win_stride
    W = sampling.n_windows(D, G, stride) if n_use is None else min(n_use, sampling.n_windows(D, G, stride))
    imgs = np.zeros((S, W, G, H, H), np.uint8)
    wmask = np.zeros((S, W), bool)
    rng = rng if rng is not None else np.random.default_rng(0)
    for s in range(S):
        if not cache.slot[i, s]:
            continue
        starts = sampling.window_starts(D, G, stride, W, train, rng)
        win, wv = sampling.gather_windows(cache.images[i, s], cache.valid[i, s], starts, G)
        if train and aug:
            win = sampling.augment_slot(win, rng, config.AUG_ROT_DEG, config.AUG_SCALE, config.AUG_SHIFT, config.AUG_INTENSITY)
        n = len(starts)
        imgs[s, :n], wmask[s, :n] = win, wv
    slot = (np.asarray(cache.slot[i]) > 0) & wmask.any(axis=1)
    return imgs, slot.astype(np.uint8), wmask


def label_arrays(df, studies_index):
    """Targets/weights as dense float32 arrays aligned to df rows (no pandas in the hot loop).

    Missing label (NaN) -> target 0 and weight 0 (masked), fixing the NaN-loss bug (98.7% of studies are unlabeled).
    An optional '<target>_weight' column scales labelled entries (e.g. report-extractor confidence)."""
    Y = np.zeros((len(df), len(config.TARGETS)), np.float32)
    Wt = np.zeros_like(Y)
    for j, t in enumerate(config.TARGETS):
        if t not in df.columns:
            continue
        y = df[t].astype(float).values
        lab = np.isfinite(y)
        Y[:, j] = np.where(lab, y, 0.0)
        w = df[f'{t}_weight'].astype(float).values if f'{t}_weight' in df.columns else np.ones(len(df))
        Wt[:, j] = np.where(lab, np.nan_to_num(w, nan=1.0), 0.0)
    rows = np.array([studies_index[s] for s in df['StudyInstanceUID']], dtype=np.int64)
    return rows, Y, Wt
''')

In [ ]:
# ---- src/preprocess/runner.py
write_module('preprocess/runner.py', r'''"""Library-level entry points used by the notebook: index -> slots/laterality -> cache -> QC.

Everything is checkpointed/resumable; every step prints the numbers that serve as acceptance gates."""
import os
import json
import time
import shutil
import tempfile

import numpy as np
import pandas as pd

from .. import config
from . import index as pix, slots as pslots, pipeline, cache as pcache, qc


# ----------------------------------------------------------------------------- environment
def pick_scratch(candidates=None, verbose=True):
    """Largest writable scratch disk OUTSIDE /kaggle/working (which is capped at ~20 GB and persisted as output)."""
    cands = candidates or ['/kaggle/temp', '/kaggle/tmp', '/tmp', tempfile.gettempdir()]
    best, table = None, []
    for c in dict.fromkeys(cands):
        try:
            if not os.path.isdir(c) or not os.access(c, os.W_OK):      # only existing, writable dirs (no side effects)
                continue
            free = shutil.disk_usage(c).free / 1e9
            table.append((c, round(free, 1)))
            if best is None or free > best[1]:
                best = (c, free)
        except OSError:
            continue
    if verbose:
        print('scratch candidates (path, free GB):', table)
    if best is None:
        raise OSError('no writable scratch directory found')
    return best[0]


def estimate_cache_gb(n_studies, cfg):
    return n_studies * config.N_SLOTS * cfg.stack_depth * cfg.img_size ** 2 / 1e9


def fit_cache_cfg(n_studies, free_gb, preset='v2', margin=0.92):
    """Largest standard (img_size, stack_depth) that fits the free disk. The 'public' preset is fixed (D=12, 336 px)."""
    if preset == 'public':
        return config.get_cfg('public')
    ladder = [(336, 24), (336, 20), (320, 20), (288, 24), (288, 20), (256, 20), (256, 16), (224, 16)]
    for img, depth in ladder:
        cfg = config.get_cfg(preset, img_size=img, stack_depth=depth)
        if estimate_cache_gb(n_studies, cfg) <= free_gb * margin:
            return cfg
    return config.get_cfg(preset, img_size=224, stack_depth=12)


# ----------------------------------------------------------------------------- 1. index
def run_index(root, out_dir, splits=('train', 'test'), workers=None, limit=0, preset='v2', progress=True):
    """Scan every series on disk, join CSV flags, assign slots, resolve laterality. Saves to out_dir:
    index.pkl, slots.pkl, sides.json, series_table.csv.gz, report.json. -> dict(ann, tab, sides, report)."""
    cfg = config.get_cfg(preset)
    os.makedirs(out_dir, exist_ok=True)
    t0 = time.time()
    df = pix.build_index(root, tuple(splits), workers, 1000, os.path.join(out_dir, 'chunks'), limit, progress=progress)
    df = pix.attach_csv_flags(df, root)
    rep = pix.index_report(df)
    rep['scan_minutes'] = round((time.time() - t0) / 60, 2)
    ann, tab = pslots.assign_all(df, cfg.slot_prefer_2d, cfg.slot_fs_priority, cfg.slot_csv_fallback)
    sides = pipeline.study_sides(df)
    rep['slot_coverage_pct'] = (tab.notna().mean() * 100).round(1).to_dict()
    rep['slots_filled_per_study'] = {int(k): int(v) for k, v in tab.notna().sum(axis=1).value_counts().sort_index().items()}
    rep['laterality'] = qc.laterality_report(sides)
    rep['header_vs_csv_fatsat_conflicts'] = int(ann['fs_conflict'].sum())
    pix.save_index(ann, os.path.join(out_dir, 'index.pkl'))
    pd.to_pickle(tab, os.path.join(out_dir, 'slots.pkl'))
    json.dump(sides, open(os.path.join(out_dir, 'sides.json'), 'w'))
    ann.drop(columns=['ordered_files', 'positions']).to_csv(os.path.join(out_dir, 'series_table.csv.gz'), index=False)
    json.dump(rep, open(os.path.join(out_dir, 'report.json'), 'w'), indent=1, default=str)
    lat = rep['laterality']
    print(json.dumps(rep, indent=1, default=str))
    print('\nGATES (index):')
    print(f"  series with read errors          : {rep['series_with_error']}  (expect ~0)")
    print(f"  non-canonical orientation series : {rep.get('non_canonical_orientation')}  (expect 0)")
    print(f"  CSV plane vs DICOM geometry      : {rep.get('csv_plane_vs_geometry_mismatch')} mismatches (expect 0)")
    print(f"  laterality unresolved            : {lat['unresolved_frac']:.2%}  (gate < 1%)")
    print(f"  laterality tag-vs-geometry clash : {lat['tag_geometry_disagree']} studies  (gate < 1% of tagged)")
    return dict(ann=ann, tab=tab, sides=sides, report=rep)


def get_index(root, out_dir, splits=('train', 'test'), workers=None, force=False, **kw):
    """Load the saved index if present (restart-friendly), else build it. -> dict(ann, tab, sides[, report])."""
    if not force and os.path.exists(os.path.join(out_dir, 'index.pkl')) and os.path.exists(os.path.join(out_dir, 'slots.pkl')):
        ann, tab, sides = load_index_dir(out_dir)
        print(f'loaded existing index: {len(ann):,} series, {len(tab):,} studies  ({out_dir})')
        return dict(ann=ann, tab=tab, sides=sides, report=json.load(open(os.path.join(out_dir, 'report.json'))))
    return run_index(root, out_dir, splits, workers, **kw)


def load_index_dir(out_dir):
    return (pix.load_index(os.path.join(out_dir, 'index.pkl')), pd.read_pickle(os.path.join(out_dir, 'slots.pkl')),
            json.load(open(os.path.join(out_dir, 'sides.json'))))


# ----------------------------------------------------------------------------- 2. cache
def run_cache(index, split, prefix, preset='v2', workers=None, limit_studies=0, fresh=False, cfg=None, studies=None,
              **cfg_overrides):
    """Build/resume the memmap cache for a split. `index` is the run_index() dict or an index directory.
    -> (cache, stats)."""
    ann, tab, sides = (index['ann'], index['tab'], index['sides']) if isinstance(index, dict) else load_index_dir(index)
    cfg = cfg or config.get_cfg(preset, **cfg_overrides)
    if studies is None:
        studies = sorted(ann[ann['split'] == split]['StudyInstanceUID'].unique())
    if limit_studies:
        studies = studies[:limit_studies]
    need = estimate_cache_gb(len(studies), cfg)
    print(f'{len(studies)} studies -> cache {need:.1f} GB  (preset={cfg.name}, D={cfg.stack_depth}, {cfg.img_size}px) at {prefix}')
    sub = ann[ann['StudyInstanceUID'].isin(set(studies))]
    records = pipeline.index_to_records(sub)
    slot_rows = {s: tab.loc[s].to_dict() for s in studies if s in tab.index}
    cache, stats = pcache.build_cache(prefix, studies, slot_rows, records, sides, cfg, workers, resume=not fresh)
    print(json.dumps(stats, indent=1))
    return cache, stats


# ----------------------------------------------------------------------------- 3. QC
def run_qc(prefix, out_dir=None, n=300, montage=True):
    """Acceptance gates on a built cache (+ optional montage PNG: one row per slot type)."""
    c = pcache.StudyCache(prefix)
    rep = qc.cache_sanity(c, n)
    print(json.dumps(rep, indent=1, default=str))
    bad = sum(rep.get(k, 0) for k in ('constant_slot', 'flagged_but_blank', 'unflagged_but_filled'))
    print(f"\nGATES (cache): constant/blank/unflagged slots = {bad} (expect 0); "
          f"mean valid depth = {rep.get('mean_valid_depth', float('nan')):.2f} (expect > 0.8)")
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
        json.dump(rep, open(os.path.join(out_dir, 'cache_sanity.json'), 'w'), indent=1, default=str)
    if montage and out_dir:
        try:
            import matplotlib
            matplotlib.use('Agg')
            import matplotlib.pyplot as plt
            rng = np.random.default_rng(0)
            done = np.where(np.asarray(c.done) == 1)[0]
            fig, ax = plt.subplots(config.N_SLOTS, 6, figsize=(14, 2.4 * config.N_SLOTS))
            for s in range(config.N_SLOTS):
                cand = [i for i in done if c.slot[i, s]]
                pick = rng.choice(cand, min(6, len(cand)), replace=False) if cand else []
                for j in range(6):
                    ax[s, j].axis('off')
                for j, i in enumerate(pick):
                    ax[s, j].imshow(c.images[i, s, c.images.shape[2] // 2], cmap='gray', vmin=0, vmax=255)
                    ax[s, j].set_title(config.SLOTS[s][0], fontsize=7)
            plt.tight_layout()
            path = os.path.join(out_dir, 'montage.png')
            plt.savefig(path, dpi=90)
            plt.close('all')
            print('montage ->', path)
        except Exception as e:                                 # a plotting problem must never fail the pipeline
            print('montage skipped:', type(e).__name__, e)
    return rep
''')

In [ ]:
# ---- src/preprocess/bench.py
write_module('preprocess/bench.py', r'''"""Calibration: measure real throughput on a SMALL sample of the actual dataset, then project the full cost.

Run it first in a fresh Kaggle session (takes ~1-3 minutes): it tells you how long the index scan and the cache build
will take on THIS machine/filesystem, how much scratch disk the cache needs, and whether the training loop's data
loading can keep a GPU fed.
"""
import os
import time

import numpy as np
import pandas as pd

from .. import config
from . import index as pix, slots as pslots, pipeline, cache as pcache, loader, runner


def _items_for(root, split, studies):
    base = os.path.join(root, pix.SPLIT_DIRS[split])
    items = []
    for st in studies:
        for e in sorted(os.scandir(os.path.join(base, st)), key=lambda x: x.name):
            if e.is_dir():
                items.append((split, st, e.name, e.path))
    return items


def calibrate(root, split='train', n_studies=24, workers=None, preset='v2', scratch=None, step_seconds=1.0,
              batch_size=config.BATCH_SIZE, n_use=4):
    from concurrent.futures import ProcessPoolExecutor
    import multiprocessing as mp
    cfg = config.get_cfg(preset)
    workers = workers or max(2, (os.cpu_count() or 4))
    scratch = scratch or runner.pick_scratch(verbose=False)
    base = os.path.join(root, pix.SPLIT_DIRS[split])
    studies = sorted(e.name for e in os.scandir(base) if e.is_dir())[:n_studies]
    items = _items_for(root, split, studies)
    n_total_series = n_total_studies = None
    for name, attr in (('train_series.csv' if split == 'train' else 'test_series.csv', 'series'), ('train.csv' if split == 'train' else 'test.csv', 'studies')):
        p = os.path.join(root, name)
        if os.path.exists(p):
            n = len(pd.read_csv(p, usecols=[0]))
            if attr == 'series':
                n_total_series = n
            else:
                n_total_studies = n
    n_total_series = n_total_series or len(items) * 4407 // max(len(studies), 1)
    n_total_studies = n_total_studies or 4407
    res = {}
    # ---- 1. index scan
    t = time.time()
    try:
        ex = ProcessPoolExecutor(max_workers=workers, mp_context=mp.get_context('fork'))
        rows = list(ex.map(pix.scan_series, items, chunksize=4))
        ex.shutdown()
    except Exception:
        rows = [pix.scan_series(i) for i in items]
    dt = time.time() - t
    df = pix.attach_csv_flags(pd.DataFrame(rows), root)
    files = int(df['n_files'].sum())
    res['index_s_per_series'] = dt / len(items)
    res['index_files_per_s'] = files / dt
    res['index_minutes_full'] = res['index_s_per_series'] * n_total_series / 60
    # ---- 2. cache build
    ann, tab = pslots.assign_all(df, cfg.slot_prefer_2d, cfg.slot_fs_priority, cfg.slot_csv_fallback)
    sides = pipeline.study_sides(df)
    records = pipeline.index_to_records(ann)
    slot_rows = {s: tab.loc[s].to_dict() for s in studies if s in tab.index}
    prefix = os.path.join(scratch, 'calib_cache')
    cache, st = pcache.build_cache(prefix, studies, slot_rows, records, sides, cfg, workers, resume=False, progress=False)
    res['cache_s_per_study_wall'] = st['seconds'] / len(studies)
    res['cache_cpu_s_per_study'] = st['seconds'] * workers / len(studies)
    res['cache_minutes_full'] = res['cache_s_per_study_wall'] * n_total_studies / 60
    res['cache_gb_full'] = runner.estimate_cache_gb(n_total_studies, cfg)
    # ---- 3. training loop data rate (single process, with augmentation)
    rng = np.random.default_rng(0)
    idx = [i for i in range(len(studies)) if cache.slot[i].any()]
    t = time.time()
    reps = 3
    for _ in range(reps):
        for i in idx:
            loader.make_sample(cache, i, cfg, True, n_use, rng, True)
    per = (time.time() - t) / (reps * len(idx))
    res['loader_ms_per_sample_1proc'] = per * 1000
    res['loader_samples_per_s_1proc'] = 1.0 / per
    need = batch_size / step_seconds
    res['loader_needed_samples_per_s'] = need
    res['loader_headroom_x'] = (1.0 / per) * min(4, workers) / need
    free = pcache.shutil.disk_usage(scratch).free / 1e9
    res['scratch_free_gb'] = free
    res['studies_sampled'] = len(studies)
    res['series_sampled'] = len(items)
    for k in ('calib_cache.images.u8', 'calib_cache.valid.u1', 'calib_cache.slot.u1', 'calib_cache.done.u1', 'calib_cache.meta.json', 'calib_cache.meta.csv'):
        try:
            os.remove(os.path.join(scratch, k))
        except OSError:
            pass
    print(f"sample: {len(studies)} studies / {len(items)} series / {files} slices on {workers} workers (preset {cfg.name})")
    print(f"  index scan   : {res['index_files_per_s']:.0f} files/s  -> full train scan ~ {res['index_minutes_full']:.1f} min")
    print(f"  cache build  : {res['cache_s_per_study_wall']:.2f} s/study wall ({res['cache_cpu_s_per_study']:.1f} cpu-s)"
          f"  -> full train cache ~ {res['cache_minutes_full']:.0f} min, {res['cache_gb_full']:.1f} GB (scratch free {free:.0f} GB)")
    print(f"  train loader : {res['loader_ms_per_sample_1proc']:.0f} ms/sample/proc -> {res['loader_headroom_x']:.1f}x headroom "
          f"over the {need:.0f} samples/s a {step_seconds:.1f}s step with batch {batch_size} needs (4 workers)")
    if res['cache_gb_full'] > free * 0.92:
        print('  !! cache does not fit the scratch disk: use runner.fit_cache_cfg(...) to pick a smaller img_size/stack_depth')
    return res
''')

In [ ]:
# ---- src/labels.py
write_module('labels.py', r'''"""Label table for training: gold labels + optional extra (report-derived / pseudo) labels with confidence weights.

Contract consumed by RSNADataset: StudyInstanceUID, the 12 TARGETS (NaN = unlabeled -> masked), optional
'<target>_weight' columns. 98.7% of train studies have no gold label, so the extra table is where most supervision
comes from; gold always wins where it exists.
"""
import os
import numpy as np
import pandas as pd

from . import config


def build_labels(root, extra_csv=None, extra_weight=0.5, out_csv=None):
    """root: competition folder. extra_csv: same schema as train.csv (targets may be soft probabilities in [0,1]),
    optionally with '<target>_weight' columns (extractor confidence). -> DataFrame (and CSV if out_csv)."""
    tr = pd.read_csv(os.path.join(root, 'train.csv'))
    out = pd.DataFrame({'StudyInstanceUID': tr['StudyInstanceUID']})
    gold = {t: (tr[t].astype(float) if t in tr.columns else pd.Series(np.nan, index=tr.index)) for t in config.TARGETS}
    ex = None
    if extra_csv and os.path.exists(extra_csv):
        ex = pd.read_csv(extra_csv).drop_duplicates('StudyInstanceUID').set_index('StudyInstanceUID').reindex(out['StudyInstanceUID'])
    out['source'] = 'none'
    any_gold = pd.concat(gold, axis=1).notna().any(axis=1).values
    out.loc[any_gold, 'source'] = 'gold'
    for t in config.TARGETS:
        g = gold[t].values
        y = g.copy()
        w = np.where(np.isfinite(g), 1.0, 0.0)
        if ex is not None and t in ex.columns:
            e = ex[t].astype(float).values
            ew = ex[f'{t}_weight'].astype(float).fillna(1.0).values if f'{t}_weight' in ex.columns else 1.0
            use = ~np.isfinite(g) & np.isfinite(e)
            y = np.where(use, e, g)
            w = np.where(use, extra_weight * ew, w)
            out.loc[use & (out['source'] == 'none'), 'source'] = 'extra'
        out[t] = y
        out[f'{t}_weight'] = w
    if out_csv:
        out.to_csv(out_csv, index=False)
    return out
''')

In [ ]:
# ---- src/dataset.py
write_module('dataset.py', r'''"""torch Dataset over the memmap cache. All real work happens in preprocess.loader (numpy)."""
import json
import numpy as np
import torch
from torch.utils.data import Dataset

from . import config
from .preprocess import cache as cache_mod
from .preprocess import loader


class RSNADataset(Dataset):
    """df: StudyInstanceUID + target columns (NaN = unlabeled) [+ '<target>_weight' columns].

    Returns (imgs uint8 [S,W,3,H,W], slot_mask [S], win_mask [S,W], targets [12], weights [12]).
    Throughput notes: the memmap is opened lazily per worker (fork-safe); labels are dense arrays; the augmentation
    RNG is seeded from (seed, epoch, index) so runs are reproducible regardless of worker scheduling."""

    def __init__(self, df, cache_prefix, cfg=None, is_train=False, n_windows_use=None, seed=config.SEED, aug=True):
        self.prefix, self.cfg = cache_prefix, cfg or cache_mod.cfg_of(cache_prefix)   # default: the cache's own config
        self.is_train, self.n_use, self.seed, self.aug, self.epoch = is_train, n_windows_use, seed, 0, 0
        with open(f'{cache_prefix}.meta.json') as fh:
            studies = json.load(fh)['studies']
        self.rows, self.Y, self.Wt = loader.label_arrays(df.reset_index(drop=True), {s: i for i, s in enumerate(studies)})
        self._cache = None

    def set_epoch(self, e):
        self.epoch = int(e)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, k):
        if self._cache is None:
            self._cache = cache_mod.StudyCache(self.prefix, 'r')
        rng = np.random.default_rng([self.seed, self.epoch, int(k)])
        imgs, slot, wm = loader.make_sample(self._cache, int(self.rows[k]), self.cfg, self.is_train, self.n_use, rng, self.aug)
        return (torch.from_numpy(imgs), torch.from_numpy(slot).float(), torch.from_numpy(wm).float(),
                torch.from_numpy(self.Y[k]), torch.from_numpy(self.Wt[k]))
''')

In [ ]:
# ---- src/model.py
write_module('model.py', r'''import os
import torch
import torch.nn as nn
from transformers import AutoModel
from . import config


class SlotHead(nn.Module):
    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden

    def forward(self, x, mask):
        # x: [B, S, dim]
        h = self.proj(x) + self.slot_emb
        att = torch.einsum('bsh,oh->bos', h, self.query) / self.hidden ** 0.5
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -10000.0).softmax(-1)
        ctx = self.drop(torch.einsum('bos,bsh->boh', att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


class WindowPool(nn.Module):
    """Masked attention pooling over the windows (2.5D triplets) of one slot: [B,S,W,dim] -> [B,S,dim]."""

    def __init__(self, dim):
        super().__init__()
        self.score = nn.Linear(dim, 1)

    def forward(self, feat, wmask):
        a = self.score(feat).squeeze(-1).masked_fill(wmask < 0.5, -10000.0).softmax(-1)
        return (a.unsqueeze(-1) * feat).sum(2)


class Model(nn.Module):
    def __init__(self, backbone, dim):
        super().__init__()
        self.backbone = backbone
        # We concatenate cls token with mean patched tokens, so dim * 2
        self.wpool = WindowPool(dim * 2)
        self.head = SlotHead(dim * 2, config.N_SLOTS, len(config.TARGETS))
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask, wmask=None):
        """imgs: uint8 [B,S,G,H,W] (one window per slot, legacy) or [B,S,W,G,H,W]; mask [B,S]; wmask [B,S,W]."""
        if imgs.dim() == 5:
            imgs = imgs.unsqueeze(2)
        B, S, W = imgs.shape[:3]
        if wmask is None:
            wmask = torch.ones(B, S, W, device=imgs.device)
        x = imgs.reshape(B * S * W, *imgs.shape[3:]).float().div_(255.0)
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state

        # Pooling: Concatenate [CLS] and Mean of patch tokens
        cls_token = out[:, 0]
        patch_tokens = out[:, 1:].mean(1)
        feat = torch.cat([cls_token, patch_tokens], dim=1).reshape(B, S, W, -1)
        slot_feat = self.wpool(feat, wmask)
        slot_mask = mask * (wmask.sum(-1) > 0).to(mask.dtype)
        return self.head(slot_feat, slot_mask)


def build_model(unfreeze_last=config.UNFREEZE_LAST, variant="dinov2-small"):
    # Note: Replace 'variant' with local path to weights if needed.
    # In the kaggle environment, this loads from local dataset directories.
    src = variant if os.path.isdir(variant) else f"facebook/{variant}"
    bb = AutoModel.from_pretrained(src)
    n_layer = len(bb.encoder.layer)

    # Freeze the entire backbone first
    for prm in bb.parameters():
        prm.requires_grad = False

    # Unfreeze the last unfreeze_last blocks
    for blk in bb.encoder.layer[max(0, n_layer - unfreeze_last):]:
        for prm in blk.parameters():
            prm.requires_grad = True

    # Unfreeze layernorm
    for prm in bb.layernorm.parameters():
        prm.requires_grad = True

    dim = bb.config.hidden_size
    return Model(bb, dim)
''')

In [ ]:
# ---- src/train.py
write_module('train.py', r'''"""Training loop on the memmap cache (data-contract fixes only; architecture/recipe unchanged).

Fixed vs the baseline: TIME_BUDGET_HOURS exists; unlabeled targets are masked (no NaN loss); loss is normalised by
the weight mass; bf16 autocast where supported (fp16+GradScaler otherwise); seeded; fold-aware validation with
per-label AUC; best-checkpoint saving; epoch-seeded augmentation.
"""
import os
import time
import dataclasses
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from . import config
from .model import build_model
from .dataset import RSNADataset
from .preprocess.cache import cfg_of as cache_cfg


def _autocast(device):
    if device.type != 'cuda':
        return torch.autocast('cpu', enabled=False), False
    use_bf16 = torch.cuda.is_bf16_supported()
    return torch.autocast('cuda', dtype=torch.bfloat16 if use_bf16 else torch.float16), (not use_bf16)


def train_epoch(model, dataloader, optimizer, scaler, scheduler, device):
    model.train()
    total, n = 0.0, 0
    for imgs, masks, wmasks, targets, weights in dataloader:
        imgs, masks, wmasks = imgs.to(device, non_blocking=True), masks.to(device), wmasks.to(device)
        targets, weights = targets.to(device), weights.to(device)
        ctx, _ = _autocast(device)
        with ctx:
            preds = model(imgs, masks, wmasks)
        loss = (F.binary_cross_entropy_with_logits(preds.float(), targets, reduction='none') * weights).sum() \
            / weights.sum().clamp_min(1.0)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total += float(loss.item())
        n += 1
    return total / max(n, 1)


@torch.no_grad()
def evaluate(model, dataloader, device):
    """-> (macro AUC over labels that have both classes, {label: auc}); weights==0 entries are ignored."""
    from sklearn.metrics import roc_auc_score
    model.eval()
    P, Y, W = [], [], []
    for imgs, masks, wmasks, targets, weights in dataloader:
        ctx, _ = _autocast(device)
        with ctx:
            p = model(imgs.to(device), masks.to(device), wmasks.to(device))
        P.append(torch.sigmoid(p.float()).cpu().numpy())
        Y.append(targets.numpy())
        W.append(weights.numpy())
    P, Y, W = np.concatenate(P), np.concatenate(Y), np.concatenate(W)
    per = {}
    for j, t in enumerate(config.TARGETS):
        m = W[:, j] > 0
        if m.sum() > 1 and 0 < Y[m, j].sum() < m.sum():
            per[t] = float(roc_auc_score(Y[m, j], P[m, j]))
    return (float(np.mean(list(per.values()))) if per else float('nan')), per


def run_training(labels_csv, cache_prefix, folds_csv=None, fold=0, out_dir='.', epochs=None,
                 n_windows_train=4, num_workers=4, seed=config.SEED, variant='dinov2-small'):
    torch.manual_seed(seed)
    np.random.seed(seed)
    cfg = cache_cfg(cache_prefix)           # the config the cache was BUILT with (never re-derive it from a preset)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    df = pd.read_csv(labels_csv)
    if folds_csv:
        df = df.merge(pd.read_csv(folds_csv)[['StudyInstanceUID', 'fold']], on='StudyInstanceUID', how='inner')
        tr, va = df[df.fold != fold], df[df.fold == fold]
    else:
        tr, va = df, df.iloc[:0]
    print(f'device={device} train={len(tr)} val={len(va)} preset={cfg.name} D={cfg.stack_depth} img={cfg.img_size}')
    model = build_model(variant=variant).to(device)      # variant may be a local folder (offline Kaggle dataset)
    ds_tr = RSNADataset(tr, cache_prefix, cfg, True, n_windows_train, seed)
    dl_tr = DataLoader(ds_tr, batch_size=config.BATCH_SIZE, shuffle=True, drop_last=len(tr) > config.BATCH_SIZE,
                       num_workers=num_workers, pin_memory=device.type == 'cuda',
                       persistent_workers=num_workers > 0, prefetch_factor=4 if num_workers > 0 else None)
    dl_va = None
    if len(va):
        ds_va = RSNADataset(va, cache_prefix, cfg, False, None, seed, aug=False)
        dl_va = DataLoader(ds_va, batch_size=max(1, config.BATCH_SIZE // 2), shuffle=False, num_workers=num_workers,
                           pin_memory=device.type == 'cuda')
    opt = torch.optim.AdamW([
        {'params': [p for p in model.backbone.parameters() if p.requires_grad], 'lr': config.LR_BACKBONE},
        {'params': list(model.wpool.parameters()) + list(model.head.parameters()), 'lr': config.LR_HEAD},
    ], weight_decay=config.WEIGHT_DECAY)
    n_ep = epochs or config.EPOCHS
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[config.LR_BACKBONE, config.LR_HEAD],
                                                total_steps=n_ep * max(len(dl_tr), 1), pct_start=0.15)
    _, need_scaler = _autocast(device)
    scaler = torch.amp.GradScaler('cuda', enabled=need_scaler)
    os.makedirs(out_dir, exist_ok=True)
    best, t0 = -1.0, time.time()
    for ep in range(n_ep):
        ds_tr.set_epoch(ep)
        loss = train_epoch(model, dl_tr, opt, scaler, sched, device)
        msg = f'epoch {ep + 1}/{n_ep} loss {loss:.4f}'
        score = float('nan')
        if dl_va is not None:
            score, per = evaluate(model, dl_va, device)
            msg += f' | val macro-AUC {score:.4f}'
        print(msg, flush=True)
        if dl_va is None or (np.isfinite(score) and score > best):
            best = score if dl_va is not None else best
            torch.save(dict(model=model.state_dict(), cfg=dataclasses.asdict(cfg), fold=fold, epoch=ep, val=score),
                       os.path.join(out_dir, f'fold{fold}_best.pt'))
        if time.time() - t0 > config.TIME_BUDGET_HOURS * 3600:
            print('Time budget reached; stopping early.')
            break
    return best
''')

In [ ]:
# ---- src/inference.py
write_module('inference.py', r'''"""Pipelined inference: CPU workers build the cache for chunk k+1 while the GPU predicts chunk k.

Hidden-rerun rules (the public notebooks' failure history): nothing here may raise on data conditions. Missing
series/flags/metadata degrade to masked slots; studies that end up with no usable slot fall back to the median
prediction of the other studies instead of a constant column; the submission is always written.
"""
import os
import tempfile
import numpy as np
import pandas as pd
import torch

from . import config
from .preprocess import index as pix
from .preprocess import slots as pslots
from .preprocess import pipeline, cache as pcache, loader


def rank_ensemble(dino_df, coatnet_df):
    """
    0.943 SOTA Rank Ensembling logic. Converts raw probabilities to percentiles (ranks),
    then applies target-specific blending weights.
    """
    coatnet_df = coatnet_df.set_index('StudyInstanceUID').reindex(dino_df['StudyInstanceUID']).reset_index()
    dino_ranks = dino_df[config.TARGETS].rank(method='average', pct=True)
    coat_ranks = coatnet_df[config.TARGETS].rank(method='average', pct=True)

    coat_weights = {label: 0.60 for label in config.TARGETS}
    coat_weights.update({
        'ACL': 0.75,
        'Medial Meniscus': 0.80,
        'Lateral Meniscus': 1.00,
        'Lateral OA': 0.75,
        'Fracture': 0.75,
    })

    blend_df = dino_df.copy()
    for label in config.TARGETS:
        w = coat_weights[label]
        blend_df[label] = ((1.0 - w) * dino_ranks[label]) + (w * coat_ranks[label])

    blend_df[config.TARGETS] = blend_df[config.TARGETS].rank(method='average', pct=True)
    return blend_df


def prepare_test_tables(root, cfg, workers=None):
    """Directory-truth index of test_series -> annotated series table, slot table, laterality, records."""
    idx = pix.build_index(root, ('test',), workers=workers, chunk=500, progress=False)
    idx = pix.attach_csv_flags(idx, root)
    idx['n_slices'] = idx['n_slices'].fillna(0) if 'n_slices' in idx else 0
    ann, tab = pslots.assign_all(idx, cfg.slot_prefer_2d, cfg.slot_fs_priority, cfg.slot_csv_fallback)
    return ann, tab, pipeline.study_sides(idx), pipeline.index_to_records(ann)


@torch.no_grad()
def predict_chunk(models, cache, a, b, cfg, device, n_use=None, batch=4):
    out = []
    for lo in range(a, b, batch):
        rows = list(range(lo, min(lo + batch, b)))
        smp = [loader.make_sample(cache, i, cfg, False, n_use) for i in rows]
        imgs = torch.from_numpy(np.stack([s[0] for s in smp])).to(device)
        slot = torch.from_numpy(np.stack([s[1] for s in smp])).float().to(device)
        wm = torch.from_numpy(np.stack([s[2] for s in smp])).float().to(device)
        ps = []
        for m in models:
            with torch.autocast('cuda', enabled=device.type == 'cuda'):
                ps.append(torch.sigmoid(m(imgs, slot, wm).float()).cpu().numpy())
        out.append(np.mean(ps, axis=0))
    return np.concatenate(out)


def run_inference(root, models, test_csv=None, cfg=None, out_csv='submission.csv', cache_dir=None, workers=None,
                  chunk=64, n_use=None, device=None):
    cfg = cfg or config.get_cfg('v2')
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ann, tab, sides, records = prepare_test_tables(root, cfg, workers)
    studies = list(tab.index)
    if test_csv and os.path.exists(test_csv):                           # keep the competition's study order
        order = pd.read_csv(test_csv)['StudyInstanceUID'].tolist()
        studies = [s for s in order if s in set(studies)] + [s for s in studies if s not in set(order)]
    cache_dir = cache_dir or ('/kaggle/temp' if os.path.isdir('/kaggle/temp') else tempfile.gettempdir())
    prefix = os.path.join(cache_dir, 'test_cache')
    slot_rows = {s: tab.loc[s].to_dict() for s in studies}
    state = {'moved': False}
    preds = np.full((len(studies), len(config.TARGETS)), np.nan, np.float32)

    def on_ready(cache, a, b):                                          # runs while workers build later chunks
        if not state['moved']:      # workers were forked at the first submit, i.e. before any CUDA call in this process
            for m in models:
                m.eval().to(device)
            state['moved'] = True
        preds[a:b] = predict_chunk(models, cache, a, b, cfg, device, n_use)

    cache, stats = pcache.build_cache(prefix, studies, slot_rows, records, sides, cfg, workers=workers, resume=False,
                                      order='seq', chunk=chunk, on_ready=on_ready)
    print('cache stats:', stats)
    empty = np.asarray(cache.slot).sum(1) == 0
    fill = np.nanmedian(preds[~empty], axis=0) if (~empty).any() else np.full(len(config.TARGETS), 0.5)
    preds[empty] = fill                                                 # never a constant column from empty studies
    preds = np.where(np.isfinite(preds), preds, fill)
    sub = pd.DataFrame(preds, columns=config.TARGETS)
    sub.insert(0, 'StudyInstanceUID', studies)
    sub.to_csv(out_csv, index=False)
    return sub, stats
''')

## Run
Run the cells below in order. Every step prints its **gates**; do not move on if a gate fails.

In [ ]:
# ---- 1. setup & environment plan ---------------------------------------------------------------------------------
import importlib, shutil, time, json
importlib.invalidate_caches()
sys.path.insert(0, BASE)
import numpy as np, pandas as pd
from src import config, labels
from src.preprocess import runner, index as pix, splits, bench, cache as pcache, loader

PRESET = 'v2'        # 'public' = bit-compatible re-implementation of the public 0.943 reader (use for its weights / A-B ablation)
ROOT = pix.discover_root()
WORK = os.environ.get('KNEE_WORK', '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.abspath('.'))
IDX_DIR, QC_DIR = f'{WORK}/knee_idx', f'{WORK}/knee_qc'
SCRATCH = runner.pick_scratch()
N_TRAIN = len(pd.read_csv(f'{ROOT}/train.csv', usecols=[0]))
FREE_GB = shutil.disk_usage(SCRATCH).free / 1e9
CFG = runner.fit_cache_cfg(N_TRAIN, FREE_GB, PRESET)          # largest standard size that fits the scratch disk
CACHE_PREFIX = f'{SCRATCH}/knee_train'
try:
    ram = {l.split(':')[0]: int(l.split()[1]) for l in open('/proc/meminfo')}
    print(f"CPU cores {os.cpu_count()} | RAM {ram['MemTotal']/1e6:.0f} GB total, {ram['MemAvailable']/1e6:.0f} GB free")
except Exception:
    pass
print(f'competition root : {ROOT}')
print(f'scratch          : {SCRATCH}  ({FREE_GB:.0f} GB free)')
print(f'cache config     : preset={CFG.name}  {CFG.img_size}px  {CFG.stack_depth} slices/slot  -> '
      f'{runner.estimate_cache_gb(N_TRAIN, CFG):.1f} GB for {N_TRAIN} studies')
if (CFG.img_size, CFG.stack_depth) != (config.get_cfg(PRESET).img_size, config.get_cfg(PRESET).stack_depth) and PRESET != 'public':
    print('NOTE: scratch disk is smaller than the full-quality cache; a reduced (img_size, stack_depth) was chosen automatically.')

In [ ]:
# ---- 2. CALIBRATION (about 1-3 minutes): measures THIS machine / filesystem on a small sample ---------------------
CAL = bench.calibrate(ROOT, 'train', n_studies=24, preset=PRESET, scratch=SCRATCH)

In [ ]:
# ---- 3. INDEX + SLOTS + LATERALITY (resumable; reloads automatically after a restart) ----------------------------
# Gates printed at the end. Reference values measured on the real metadata of all 4,407 train studies:
#   slot coverage % ~ SAG_FLUID_FS 94.3 | COR_FLUID_FS 95.5 | AX_FLUID_FS 99.4 | SAG_FLUID_NOFS 62.6 | COR_T1 64.1 | SAG_T1 42.5
FORCE_REINDEX = False
IDX = runner.get_index(ROOT, IDX_DIR, ('train', 'test'), force=FORCE_REINDEX)

In [ ]:
# ---- 4. LABELS + FOLDS --------------------------------------------------------------------------------------------
# EXTRA_LABELS_CSV: optional report-derived / pseudo labels, same schema as train.csv (12 target columns, soft
# probabilities allowed, optional '<target>_weight' confidence columns). Gold labels always win where they exist.
EXTRA_LABELS_CSV = None            # e.g. '/kaggle/input/<your-label-dataset>/labels.csv'
EXTRA_WEIGHT = 0.5                 # loss weight multiplier for extra labels
lab = labels.build_labels(ROOT, EXTRA_LABELS_CSV, EXTRA_WEIGHT, out_csv=f'{WORK}/labels.csv')
print('label sources:', lab['source'].value_counts().to_dict())
tr = pd.read_csv(f'{ROOT}/train.csv')
sm = splits.make_study_meta(IDX['ann'], tr, lab)
for scheme in ('site', 'group'):
    f = splits.group_folds(sm, 5, config.SEED, scheme=scheme)
    f.to_csv(f'{WORK}/folds_{scheme}.csv', index=False)
    poor = sm.set_index('StudyInstanceUID')['poor'].reindex(f['StudyInstanceUID']).fillna(False).values
    print(f"folds[{scheme:5s}] sizes {f['fold'].value_counts().sort_index().to_dict()} | "
          f"metadata-poor studies/fold {pd.Series(poor).groupby(f['fold'].values).sum().astype(int).to_dict()}")
print('use folds_site.csv for model selection; folds_group.csv is the unseen-scanner robustness check')

In [ ]:
# ---- 5. BUILD THE TRAIN CACHE (CPU only; do this BEFORE any GPU/torch work) ----------------------------------------
# Resumable: if the session dies, run this cell again and it continues where it stopped.
CACHE, STATS = runner.run_cache(IDX, 'train', CACHE_PREFIX, cfg=CFG)

In [ ]:
# ---- 6. QC GATES + montage ----------------------------------------------------------------------------------------
QC = runner.run_qc(CACHE_PREFIX, QC_DIR, n=300)
try:
    from IPython.display import Image, display
    display(Image(f'{QC_DIR}/montage.png'))
except Exception:
    pass

In [ ]:
# ---- 7. LOADER THROUGHPUT (needs torch): can the data feed the GPU? -----------------------------------------------
try:
    import torch
    from torch.utils.data import DataLoader
    from src.dataset import RSNADataset
    have_torch = True
except ImportError:
    have_torch = False
    print('torch not available here - loader benchmark skipped')
if have_torch:
    lab_c = lab[lab['StudyInstanceUID'].isin(CACHE.studies)]
    ds = RSNADataset(lab_c, CACHE_PREFIX, None, True, 4)                 # cfg=None -> the cache's own config
    nw = min(4, os.cpu_count() or 1)
    dl = DataLoader(ds, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=nw, drop_last=len(ds) > config.BATCH_SIZE,
                    pin_memory=torch.cuda.is_available(), persistent_workers=nw > 0, prefetch_factor=4 if nw > 0 else None)
    t0, n, nb = time.time(), 0, 0
    for imgs, slot, wm, y, w in dl:
        n += imgs.shape[0]; nb += 1
        if nb == 1:
            print('batch:', tuple(imgs.shape), imgs.dtype, f'{imgs.numel()/1e6:.0f} MB uint8 | targets {tuple(y.shape)}')
        if nb >= 40:
            break
    dt = time.time() - t0
    print(f'{n/dt:.1f} samples/s with {nw} workers  (a 1.0 s GPU step at batch {config.BATCH_SIZE} needs {config.BATCH_SIZE:.0f}/s)')

In [ ]:
# ---- 8. TRAINING (optional; flip the flag; needs torch + transformers + DINOv2 weights) ----------------------------
RUN_TRAINING = False
FOLD = 0
VARIANT = 'dinov2-small'            # or a local folder of an attached weights dataset (no internet needed)
if RUN_TRAINING:
    from src import train
    train.run_training(f'{WORK}/labels.csv', CACHE_PREFIX, f'{WORK}/folds_site.csv', fold=FOLD,
                       out_dir=f'{WORK}/ckpt', variant=VARIANT)       # the config is read from the cache itself

In [ ]:
# ---- 9. INFERENCE on the hidden test set (optional; for the SUBMISSION notebook) --------------------------------
# CPU workers build the test cache chunk by chunk while the GPU predicts the previous chunk (pipelined).
# Use the SAME cfg as the cache the models were trained on (CFG here, or pcache.cfg_of(<train cache prefix>)).
RUN_INFERENCE = False
CKPTS = []                          # e.g. [f'{WORK}/ckpt/fold{k}_best.pt' for k in range(5)]
if RUN_INFERENCE:
    import torch
    from src import inference, model as M
    models = []
    for p in CKPTS:
        m = M.build_model(variant=VARIANT)
        m.load_state_dict(torch.load(p, map_location='cpu')['model'])
        models.append(m)
    sub, st = inference.run_inference(ROOT, models, f'{ROOT}/test.csv', CFG, f'{WORK}/submission.csv', cache_dir=SCRATCH)
    print(sub.shape, st)

## Acceptance gates (read these after the first real run)

| Gate | Expected | If it fails |
|---|---|---|
| non-canonical orientation series | 0 | check `report.json → orient_codes`; `reorient_stack` will correct, but investigate |
| CSV plane vs DICOM geometry | 0 mismatches | decide which to trust (audit: 100% agreement) |
| laterality unresolved | < 1% | the geometry rule (|x| ≥ 20 mm) is not resolving this data → lower `LAT_MIN_OFFSET_MM` or add an image-based side classifier |
| laterality tag vs geometry disagreements | < 1% of tagged studies | **the sign convention (x<0 ⇒ right) is wrong for this data → swap it in `geometry.resolve_laterality`** |
| slot coverage | ≈ 94 / 95 / 99 / 63 / 64 / 43 % | header rules not matching → inspect `series_table.csv.gz` |
| cache: constant / blank / unflagged slots | 0 | inspect `QC['failures']` and the montage |
| cache: mean valid depth | > 0.8 | z-window larger than the stacks; lower `stack_depth` or raise `z_step_mm` |
| loader headroom | ≥ 3× | fewer windows (`n_windows_use`) or more workers |

### Verification status (what was and was not checked)
* **Checked against real data (your audit tables):** slot rule reproduces the public one on 4,406/4,407 studies (the single
  difference is an exact slice-count tie, now deterministic); site-stratified folds keep every language in every fold with
  42–53 metadata-poor studies per fold; filename-order randomness (all 24,371 series).
* **Checked on a physically consistent synthetic dataset with ground truth:** slice order 35/35 correct (baseline `sorted(listdir)`: 0/35);
  limb width 109.7–111.7 mm for a 110 mm truth across normal, mirrored, 2:1 wide off-centre, non-square-pixel and padded-FOV series;
  mirrored left/right knees give **bit-identical** tensors after canonicalisation (max difference 0 vs 142 without);
  the `public` preset matches an independent reference reader on 99.9996% of 3.87 M pixels (max 1 grey level);
  cache == direct pipeline output, resume rebuilds exactly the undone studies, config mismatch raises, thread-pool fallback is identical,
  hidden-set conditions (no CSV, mangled flags, no positions, no spacing, empty/corrupt series, `test_series` path) degrade gracefully.
* **Not executable in the build sandbox (no torch):** `dataset.py`, `model.py`, `train.py`, `inference.py` — syntax- and lint-checked only; cell 7 benchmarks the Dataset on Kaggle.
* **Only the real run can confirm:** the laterality sign convention and the real-filesystem throughput (cell 2 measures it).